# 🛒 AI-Powered Ecommerce Report Generation System

> *A production-inspired prototype that automatically generates personalized Daily Digests and Weekly Reports for D2C ecommerce brands.*

---

## 1. Introduction

### What is this?

This notebook is the **complete application**. Running it from top to bottom produces:

- 📊 Synthetic ecommerce datasets for two distinct customer archetypes
- 🔍 Feature engineering with anomaly detection and movement scoring
- 🏆 A deterministic Importance Ranking Engine (no ML, fully explainable)
- 🎯 A three-tier Personalization Engine (Cold Start → Warm → Mature)
- 📋 Structured Evidence JSON objects (what the LLM actually sees)
- 🤖 AI-written Daily Digests and Weekly Reports via Claude API
- 📄 Exported Markdown + PDF reports
- 🗂️ A complete Product Design Document

### The Central Question

> **"What changed in my business today, and should I care?"**

This system answers that question differently for each customer — a growth-stage startup cares about ROAS and conversion; an established brand cares about inventory and fulfillment.

### System Architecture

```
Raw Ecommerce Data (CSV)
        │
        ▼
┌─────────────────────┐
│  Feature Engineering │  → Daily Δ%, WoW%, MoM%, Z-score, Momentum
└─────────────────────┘
        │
        ▼
┌──────────────────────┐
│ Movement Detection   │  → MetricMovement objects (severity, confidence)
└──────────────────────┘
        │
        ▼
┌──────────────────────┐
│ Importance Ranker    │  → Scored, ranked movements (0-100)
└──────────────────────┘
        │
        ▼
┌──────────────────────┐
│ Personalization      │  → Customer-weighted scores (Cold/Warm/Mature)
└──────────────────────┘
        │
        ▼
┌──────────────────────┐
│ Evidence Builder     │  → Structured JSON evidence (no raw tables)
└──────────────────────┘
        │
        ▼
┌──────────────────────┐
│  Claude API          │  → AI-written narrative reports
└──────────────────────┘
        │
        ▼
   Markdown + PDF Reports
```

### Design Philosophy

| Principle | Implementation |
|-----------|----------------|
| **Grounded AI** | LLM only summarizes evidence — never invents causes |
| **Transparent ranking** | Every importance score is fully explained |
| **Customer-first** | Two archetypes, two completely different reports |
| **Deterministic core** | Statistical engine is reproducible and auditable |
| **Production-inspired** | Mirrors Shopify Analytics / Triple Whale architecture |


---
## 2. Imports


In [ ]:
# ============================================================
# Section 2 — Imports
# ============================================================

# Standard library
import os
import sys
import json
import math
import random
import warnings
import textwrap
from pathlib import Path
from datetime import datetime, timedelta
from dataclasses import dataclass, field, asdict
from typing import Optional, List, Dict, Tuple, Any
from enum import Enum

warnings.filterwarnings('ignore')

# Data science
import plotly
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import MinMaxScaler

# Visualization
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import FancyBboxPatch
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
import plotly

# AI
import anthropic

# Environment
from dotenv import load_dotenv

# PDF
from reportlab.lib.pagesizes import A4, letter
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch, cm
from reportlab.lib import colors
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Image as RLImage,
    Table, TableStyle, HRFlowable, PageBreak
)
from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_RIGHT, TA_JUSTIFY

# Utilities
from tqdm.notebook import tqdm

# ── Version report ───────────────────────────────────────────
print(f"Python        : {sys.version.split()[0]}")
print(f"NumPy         : {np.__version__}")
print(f"Pandas        : {pd.__version__}")
print(f"Matplotlib    : {matplotlib.__version__}")
print(f"Plotly        : {plotly.__version__}")
print(f"Anthropic SDK : {anthropic.__version__}")
print("\n✅ All imports successful.")


---
## 3. Configuration


In [ ]:
# ============================================================
# Section 3 — Configuration
# ============================================================

# Load environment variables
load_dotenv()
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")
CLAUDE_MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-4-5")

if not ANTHROPIC_API_KEY:
    print("⚠️  WARNING: ANTHROPIC_API_KEY not set. AI generation will be skipped.")
    print("   Copy .env.example → .env and add your key.")
    AI_ENABLED = False
else:
    print(f"✅ Anthropic API key loaded. Model: {CLAUDE_MODEL}")
    AI_ENABLED = True

# ── Date Configuration ───────────────────────────────────────
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

END_DATE = datetime(2024, 6, 20)   # "Today" for report generation
START_DATE = END_DATE - timedelta(days=179)  # 180 days of data
N_DAYS = 180

print(f"\n📅 Data range: {START_DATE.date()} → {END_DATE.date()} ({N_DAYS} days)")

# ── Directory Configuration ──────────────────────────────────
BASE_DIR = Path(".").resolve()
DIRS = {
    "synthetic_data": BASE_DIR / "synthetic_data",
    "reports_daily":  BASE_DIR / "generated_reports" / "daily",
    "reports_weekly": BASE_DIR / "generated_reports" / "weekly",
    "charts":         BASE_DIR / "assets" / "charts",
}
for name, path in DIRS.items():
    path.mkdir(parents=True, exist_ok=True)

print("\n📁 Directory structure:")
for name, path in DIRS.items():
    print(f"   {name:<20} → {path}")

# ── Visualization Config ─────────────────────────────────────
PLOTLY_TEMPLATE = "plotly_dark"
CHART_WIDTH  = 1200
CHART_HEIGHT = 500
CHART_DPI    = 150
pio.templates.default = PLOTLY_TEMPLATE

# Colour palette
COLORS = {
    "primary":   "#6C63FF",
    "secondary": "#FF6584",
    "success":   "#43D9AD",
    "warning":   "#FFD166",
    "danger":    "#EF4444",
    "neutral":   "#94A3B8",
    "customerA": "#6C63FF",
    "customerB": "#FF6584",
}

# ── Thresholds ───────────────────────────────────────────────
SEVERITY_THRESHOLDS = {
    "CRITICAL": 3.0,   # Z-score
    "HIGH":     2.0,
    "MEDIUM":   1.5,
    "LOW":      1.0,
}

print("\n✅ Configuration complete.")


---
## 4. Synthetic Data Generation

We generate **180 days** of realistic ecommerce data for two customers.

The data includes:
- **Trend** (gradual business growth)
- **Seasonality** (weekly patterns, day-of-week effects)
- **Holiday effects** (spikes around major shopping events)
- **Random variation** (Gaussian noise)
- **Intentional anomalies** (traffic drops, marketing spikes, inventory issues)


In [ ]:
# ============================================================
# Section 4 — Synthetic Data Generation
# ============================================================

class EcommerceDataGenerator:
    """
    Generates realistic synthetic ecommerce time-series data.

    Design decisions:
    - Trend: linear growth scaled by company stage
    - Seasonality: weekly cycle + day-of-week effects
    - Anomalies: injected at specific dates for realism
    - Correlations: spend ↔ sessions ↔ revenue are correlated
    """

    def __init__(self, seed: int = 42):
        self.rng = np.random.default_rng(seed)
        self.dates = pd.date_range(START_DATE, periods=N_DAYS, freq='D')

    def _trend(self, days: int, start: float, growth_rate: float) -> np.ndarray:
        """Linear growth trend."""
        return start * (1 + growth_rate) ** np.arange(days)

    def _weekly_seasonality(self, days: int, amplitude: float = 0.15) -> np.ndarray:
        """Weekly sine wave + day-of-week multipliers."""
        t = np.arange(days)
        sine = amplitude * np.sin(2 * np.pi * t / 7)
        dow_mult = np.array([1.0, 1.05, 1.02, 0.95, 1.10, 1.20, 0.85])
        dow = np.array([dow_mult[d % 7] for d in range(days)])
        return sine + (dow - 1.0)

    def _holiday_effects(self, days: int) -> np.ndarray:
        """Inject holiday spikes."""
        effect = np.zeros(days)
        start = START_DATE
        for holiday_offset, magnitude, duration in [
            (30, 0.35, 4),   # Spring sale
            (75, 0.50, 5),   # Mid-year flash sale
            (120, 0.40, 3),  # Back to school
            (155, 0.60, 6),  # End-of-period mega sale
        ]:
            for d in range(duration):
                idx = holiday_offset + d
                if 0 <= idx < days:
                    peak = magnitude * math.sin(math.pi * (d + 0.5) / duration)
                    effect[idx] += peak
        return effect

    def _anomalies(self, series: np.ndarray, anomaly_spec: list) -> np.ndarray:
        """Inject intentional anomalies at specified indices."""
        out = series.copy()
        for idx, multiplier in anomaly_spec:
            if 0 <= idx < len(out):
                out[idx] *= multiplier
        return out

    def _noise(self, days: int, scale: float = 0.05) -> np.ndarray:
        """Gaussian noise."""
        return self.rng.normal(0, scale, days)

    def generate_customer_a(self) -> pd.DataFrame:
        """
        Customer A — GrowthCo (Startup, growth-focused).
        High volatility, strong paid-ads dependency, fast growth.
        """
        n = N_DAYS
        dates = self.dates

        trend     = self._trend(n, start=1.0, growth_rate=0.003)
        season    = self._weekly_seasonality(n, amplitude=0.20)
        holidays  = self._holiday_effects(n)
        composite = trend * (1 + season + holidays)

        # Base revenue: $8k/day growing to ~$15k
        base_revenue = 8000
        revenue_raw = base_revenue * composite * (1 + self._noise(n, 0.08))
        # Anomalies: traffic drop day 45, marketing spike day 90, crash day 140
        revenue = self._anomalies(revenue_raw, [(45, 0.45), (90, 1.80), (140, 0.30), (141, 0.55)])

        # Sessions correlated with spend
        spend_meta   = 800 * composite * (1 + self._noise(n, 0.10))
        spend_google = 400 * composite * (1 + self._noise(n, 0.10))
        spend_total  = spend_meta + spend_google
        sessions_raw = 3000 * composite * (1 + self._noise(n, 0.07))
        sessions = self._anomalies(sessions_raw, [(45, 0.40), (90, 1.75), (140, 0.28)])

        # Conversion rate: 2.5-3.5%, dips during traffic drops
        cvr = np.clip(0.03 + self._noise(n, 0.004) + 0.002 * np.sin(2*np.pi*np.arange(n)/30), 0.015, 0.055)
        cvr = self._anomalies(cvr, [(140, 0.70), (141, 0.75)])

        orders = np.maximum(1, (sessions * cvr).astype(int))
        aov    = np.where(orders > 0, revenue / orders, 65.0)
        roas   = np.where(spend_total > 0, revenue / spend_total, 0)
        cac    = np.where(orders > 0, spend_total / orders, 0)

        ctr    = np.clip(0.025 + self._noise(n, 0.003), 0.01, 0.06)
        cpm    = 12.0 + self._noise(n, 1.2) * (trend ** 0.5)
        cpc    = np.where(ctr > 0, cpm / (ctr * 1000), 0)

        email_rev = 500 * composite * (1 + self._noise(n, 0.15))
        email_open_rate = np.clip(0.22 + self._noise(n, 0.02), 0.10, 0.45)

        repeat_purchase_rate = np.clip(0.18 + self._noise(n, 0.02) + 0.0003 * np.arange(n), 0.08, 0.40)
        refund_rate          = np.clip(0.03 + self._noise(n, 0.005), 0.005, 0.12)
        checkout_abandon     = np.clip(0.68 + self._noise(n, 0.03) - 0.0002 * np.arange(n), 0.50, 0.85)

        inventory            = np.clip(500 + 10 * np.arange(n) + self._noise(n, 30), 50, 2000).astype(int)
        fulfillment_sla_days = np.clip(2.1 + self._noise(n, 0.3), 1.0, 5.0)
        shipping_delay_pct   = np.clip(0.05 + self._noise(n, 0.015), 0.0, 0.25)
        return_rate          = np.clip(0.04 + self._noise(n, 0.008), 0.01, 0.15)
        support_tickets      = np.maximum(0, (orders * 0.02 + self._noise(n, 2)).astype(int))

        df = pd.DataFrame({
            'date':                 dates,
            'customer_id':          'A',
            'revenue':              np.maximum(0, revenue),
            'orders':               orders,
            'sessions':             np.maximum(0, sessions).astype(int),
            'conversion_rate':      cvr,
            'aov':                  np.maximum(0, aov),
            'roas':                 np.maximum(0, roas),
            'cac':                  np.maximum(0, cac),
            'spend_total':          np.maximum(0, spend_total),
            'spend_meta':           np.maximum(0, spend_meta),
            'spend_google':         np.maximum(0, spend_google),
            'ctr':                  ctr,
            'cpm':                  np.maximum(0, cpm),
            'cpc':                  np.maximum(0, cpc),
            'email_revenue':        np.maximum(0, email_rev),
            'email_open_rate':      email_open_rate,
            'repeat_purchase_rate': repeat_purchase_rate,
            'refund_rate':          refund_rate,
            'checkout_abandonment': checkout_abandon,
            'inventory_units':      inventory,
            'fulfillment_sla_days': fulfillment_sla_days,
            'shipping_delay_pct':   shipping_delay_pct,
            'return_rate':          return_rate,
            'support_tickets':      support_tickets,
        })
        return df

    def generate_customer_b(self) -> pd.DataFrame:
        """
        Customer B — RetailPro (Established brand, operations-focused).
        Lower volatility, strong retention, inventory issues, fulfillment focus.
        """
        n = N_DAYS
        dates = self.dates

        trend    = self._trend(n, start=1.0, growth_rate=0.001)   # Slower growth
        season   = self._weekly_seasonality(n, amplitude=0.10)    # Less volatile
        holidays = self._holiday_effects(n)
        composite = trend * (1 + season + holidays)

        # Base revenue: $45k/day (larger, more stable)
        base_revenue = 45000
        revenue_raw = base_revenue * composite * (1 + self._noise(n, 0.04))
        # Anomalies: inventory stockout day 60-65, fulfillment crisis day 110
        revenue = self._anomalies(revenue_raw, [(60, 0.65), (61, 0.55), (62, 0.60),
                                                 (63, 0.70), (110, 0.75), (111, 0.80)])

        spend_meta   = 3000 * composite * (1 + self._noise(n, 0.06))
        spend_google = 2000 * composite * (1 + self._noise(n, 0.06))
        spend_total  = spend_meta + spend_google
        sessions_raw = 25000 * composite * (1 + self._noise(n, 0.04))
        sessions = self._anomalies(sessions_raw, [(60, 0.70), (61, 0.65), (110, 0.85)])

        cvr = np.clip(0.038 + self._noise(n, 0.003), 0.02, 0.07)

        orders = np.maximum(1, (sessions * cvr).astype(int))
        aov    = np.where(orders > 0, revenue / orders, 120.0)
        roas   = np.where(spend_total > 0, revenue / spend_total, 0)
        cac    = np.where(orders > 0, spend_total / orders, 0)

        ctr    = np.clip(0.022 + self._noise(n, 0.002), 0.01, 0.05)
        cpm    = 11.0 + self._noise(n, 1.0)
        cpc    = np.where(ctr > 0, cpm / (ctr * 1000), 0)

        email_rev = 5000 * composite * (1 + self._noise(n, 0.10))
        email_open_rate = np.clip(0.28 + self._noise(n, 0.02), 0.15, 0.50)

        # High retention — established brand
        repeat_purchase_rate = np.clip(0.42 + self._noise(n, 0.02), 0.25, 0.65)
        refund_rate          = np.clip(0.06 + self._noise(n, 0.008), 0.02, 0.18)
        # Inventory crisis → spike in refund rate
        refund_rate = self._anomalies(refund_rate, [(60, 1.8), (61, 2.0), (62, 1.9)])

        checkout_abandon = np.clip(0.62 + self._noise(n, 0.02), 0.45, 0.80)

        # Inventory: stockout event
        inv_base = 3000 - 10 * np.arange(n) + self._noise(n, 100)
        inventory = np.clip(inv_base, 0, 5000).astype(int)
        inventory[57:67] = np.clip(inventory[57:67] * 0.05, 0, 200).astype(int)  # Stockout

        # Fulfillment crisis day 110
        fulfillment_sla_days = np.clip(1.8 + self._noise(n, 0.2), 1.0, 4.0)
        fulfillment_sla_days = self._anomalies(fulfillment_sla_days, [(110, 2.5), (111, 2.2), (112, 1.8)])

        shipping_delay_pct = np.clip(0.04 + self._noise(n, 0.01), 0.0, 0.30)
        shipping_delay_pct = self._anomalies(shipping_delay_pct, [(110, 3.5), (111, 3.0)])

        return_rate     = np.clip(0.08 + self._noise(n, 0.01), 0.02, 0.20)
        return_rate     = self._anomalies(return_rate, [(60, 1.7), (61, 1.9)])

        support_tickets = np.maximum(0, (orders * 0.015 + self._noise(n, 5)).astype(int))
        support_tickets = self._anomalies(support_tickets, [(60, 3.0), (110, 2.5)])

        df = pd.DataFrame({
            'date':                 dates,
            'customer_id':          'B',
            'revenue':              np.maximum(0, revenue),
            'orders':               orders,
            'sessions':             np.maximum(0, sessions).astype(int),
            'conversion_rate':      cvr,
            'aov':                  np.maximum(0, aov),
            'roas':                 np.maximum(0, roas),
            'cac':                  np.maximum(0, cac),
            'spend_total':          np.maximum(0, spend_total),
            'spend_meta':           np.maximum(0, spend_meta),
            'spend_google':         np.maximum(0, spend_google),
            'ctr':                  ctr,
            'cpm':                  np.maximum(0, cpm),
            'cpc':                  np.maximum(0, cpc),
            'email_revenue':        np.maximum(0, email_rev),
            'email_open_rate':      email_open_rate,
            'repeat_purchase_rate': repeat_purchase_rate,
            'refund_rate':          refund_rate,
            'checkout_abandonment': checkout_abandon,
            'inventory_units':      np.maximum(0, inventory),
            'fulfillment_sla_days': fulfillment_sla_days,
            'shipping_delay_pct':   np.maximum(0, shipping_delay_pct),
            'return_rate':          return_rate,
            'support_tickets':      np.maximum(0, support_tickets).astype(int),
        })
        return df


# Generate data
gen = EcommerceDataGenerator(seed=RANDOM_SEED)
df_a = gen.generate_customer_a()
df_b = gen.generate_customer_b()
df_all = pd.concat([df_a, df_b], ignore_index=True)

# Save CSVs
df_all.to_csv(DIRS['synthetic_data'] / 'ecommerce_data.csv', index=False)
print(f"✅ ecommerce_data.csv saved — {len(df_all)} rows, {len(df_all.columns)} columns")

# Customer profile CSV
customers_data = pd.DataFrame([
    {'customer_id': 'A', 'name': 'GrowthCo', 'stage': 'startup',
     'size': 'small', 'industry': 'fashion', 'founded_years': 2,
     'priority_1': 'revenue', 'priority_2': 'roas', 'priority_3': 'conversion_rate'},
    {'customer_id': 'B', 'name': 'RetailPro', 'stage': 'established',
     'size': 'medium', 'industry': 'home_goods', 'founded_years': 12,
     'priority_1': 'inventory_units', 'priority_2': 'return_rate', 'priority_3': 'fulfillment_sla_days'},
])
customers_data.to_csv(DIRS['synthetic_data'] / 'customers.csv', index=False)
print(f"✅ customers.csv saved")

print(f"\n📊 Customer A (GrowthCo) — Revenue range:  ${df_a.revenue.min():,.0f} – ${df_a.revenue.max():,.0f}")
print(f"📊 Customer B (RetailPro) — Revenue range: ${df_b.revenue.min():,.0f} – ${df_b.revenue.max():,.0f}")
df_a.tail(5)[['date','revenue','roas','conversion_rate','sessions']]


---
## 5. Exploratory Analysis


In [ ]:
# ============================================================
# Section 5 — Exploratory Analysis
# ============================================================

def save_plotly_chart(fig: go.Figure, name: str) -> Path:
    """Save a Plotly figure as PNG for PDF embedding."""
    path = DIRS['charts'] / f"{name}.png"
    try:
        fig.write_image(str(path), width=CHART_WIDTH, height=CHART_HEIGHT, scale=1.5)
    except Exception as e:
        print(f"  ⚠️  Could not save {name}.png (kaleido issue): {e}")
    return path

CHART_PATHS: Dict[str, Path] = {}

# ── Chart 1: Revenue Trend (180d) ────────────────────────────
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_a['date'], y=df_a['revenue'],
    name='GrowthCo (A)', line=dict(color=COLORS['customerA'], width=2),
    fill='tozeroy', fillcolor='rgba(108,99,255,0.08)'
))

# Annotate anomalies
anomaly_dates_a = [START_DATE + timedelta(days=d) for d in [45, 90, 140]]
anomaly_labels_a = ['Traffic Drop', 'Marketing Spike', 'Conversion Crash']
for dt, lbl in zip(anomaly_dates_a, anomaly_labels_a):
    rev_at = df_a.loc[df_a['date']==pd.Timestamp(dt), 'revenue']
    if len(rev_at):
        fig.add_annotation(
            x=dt, y=rev_at.values[0],
            text=lbl, showarrow=True, arrowhead=2,
            font=dict(color=COLORS['warning'], size=10),
            arrowcolor=COLORS['warning'], ax=0, ay=-40
        )

fig.update_layout(
    title='GrowthCo — Revenue Trend (180 Days)',
    xaxis_title='Date', yaxis_title='Revenue ($)',
    height=CHART_HEIGHT, width=CHART_WIDTH,
    yaxis_tickformat='$,.0f'
)
CHART_PATHS['revenue_trend_a'] = save_plotly_chart(fig, 'revenue_trend_a')
fig.show()


In [ ]:
# ── Chart 2: Both Customers Revenue Comparison ──────────────
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=['GrowthCo (A) — Startup', 'RetailPro (B) — Established Brand'],
                    vertical_spacing=0.08)

# 7-day rolling average
rev_a_roll = df_a['revenue'].rolling(7).mean()
rev_b_roll = df_b['revenue'].rolling(7).mean()

fig.add_trace(go.Scatter(x=df_a['date'], y=df_a['revenue'], name='GrowthCo Daily',
                          line=dict(color=COLORS['customerA'], width=1), opacity=0.5), row=1, col=1)
fig.add_trace(go.Scatter(x=df_a['date'], y=rev_a_roll, name='GrowthCo 7d MA',
                          line=dict(color=COLORS['customerA'], width=2.5)), row=1, col=1)

fig.add_trace(go.Scatter(x=df_b['date'], y=df_b['revenue'], name='RetailPro Daily',
                          line=dict(color=COLORS['customerB'], width=1), opacity=0.5), row=2, col=1)
fig.add_trace(go.Scatter(x=df_b['date'], y=rev_b_roll, name='RetailPro 7d MA',
                          line=dict(color=COLORS['customerB'], width=2.5)), row=2, col=1)

fig.update_layout(title='Revenue Comparison: Startup vs Established Brand',
                  height=700, width=CHART_WIDTH, showlegend=True)
fig.update_yaxes(tickformat='$,.0f')
CHART_PATHS['revenue_comparison'] = save_plotly_chart(fig, 'revenue_comparison')
fig.show()


In [ ]:
# ── Chart 3: ROAS Trend ──────────────────────────────────────
fig = go.Figure()
for df_cust, name, color in [(df_a, 'GrowthCo', COLORS['customerA']),
                               (df_b, 'RetailPro', COLORS['customerB'])]:
    roll = df_cust['roas'].rolling(7).mean()
    fig.add_trace(go.Scatter(x=df_cust['date'], y=df_cust['roas'],
                              name=f'{name} Daily', line=dict(color=color, width=1), opacity=0.4))
    fig.add_trace(go.Scatter(x=df_cust['date'], y=roll,
                              name=f'{name} 7d ROAS', line=dict(color=color, width=2.5)))
fig.add_hline(y=4.0, line_dash='dot', line_color=COLORS['warning'],
              annotation_text='Target ROAS: 4.0', annotation_position='right')
fig.update_layout(title='ROAS Trend — Both Customers', yaxis_title='ROAS',
                  height=CHART_HEIGHT, width=CHART_WIDTH)
CHART_PATHS['roas_trend'] = save_plotly_chart(fig, 'roas_trend')
fig.show()


In [ ]:
# ── Chart 4: CAC Trend ──────────────────────────────────────
fig = go.Figure()
for df_cust, name, color in [(df_a, 'GrowthCo', COLORS['customerA']),
                               (df_b, 'RetailPro', COLORS['customerB'])]:
    roll = df_cust['cac'].rolling(7).mean()
    fig.add_trace(go.Scatter(x=df_cust['date'], y=roll, name=f'{name} CAC (7d MA)',
                              line=dict(color=color, width=2.5)))
fig.update_layout(title='Customer Acquisition Cost Trend', yaxis_title='CAC ($)',
                  height=CHART_HEIGHT, width=CHART_WIDTH, yaxis_tickformat='$,.2f')
CHART_PATHS['cac_trend'] = save_plotly_chart(fig, 'cac_trend')
fig.show()


In [ ]:
# ── Chart 5: Correlation Heatmap ────────────────────────────
metrics_for_corr = ['revenue','sessions','conversion_rate','aov','roas','cac',
                    'spend_total','ctr','email_revenue','repeat_purchase_rate','return_rate']

corr_a = df_a[metrics_for_corr].corr()
fig = px.imshow(corr_a, color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
                title='GrowthCo — Metric Correlation Heatmap',
                text_auto='.2f', aspect='auto')
fig.update_layout(height=600, width=CHART_WIDTH)
CHART_PATHS['corr_heatmap'] = save_plotly_chart(fig, 'corr_heatmap')
fig.show()


In [ ]:
# ── Chart 6: Weekly Revenue Bar Chart ───────────────────────
df_a_weekly = df_a.copy()
df_a_weekly['week'] = df_a_weekly['date'].dt.to_period('W').dt.start_time
weekly_rev_a = df_a_weekly.groupby('week')['revenue'].sum().reset_index()

fig = go.Figure(go.Bar(
    x=weekly_rev_a['week'], y=weekly_rev_a['revenue'],
    marker_color=COLORS['customerA'],
    marker_line_color=COLORS['primary'], marker_line_width=0.5
))
fig.update_layout(title='GrowthCo — Weekly Revenue', yaxis_title='Revenue ($)',
                  height=CHART_HEIGHT, width=CHART_WIDTH, yaxis_tickformat='$,.0f')
CHART_PATHS['weekly_revenue_a'] = save_plotly_chart(fig, 'weekly_revenue_a')
fig.show()


In [ ]:
# ── Chart 7: RetailPro Inventory & Fulfillment Crisis ───────
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=['Inventory Units', 'Fulfillment SLA (days)'],
                    vertical_spacing=0.08)
fig.add_trace(go.Scatter(x=df_b['date'], y=df_b['inventory_units'],
                          name='Inventory', line=dict(color=COLORS['customerB'], width=2),
                          fill='tozeroy', fillcolor='rgba(255,101,132,0.1)'), row=1, col=1)
fig.add_hline(y=100, line_dash='dash', line_color=COLORS['danger'],
              annotation_text='Critical Stock Level', row=1, col=1)
fig.add_trace(go.Scatter(x=df_b['date'], y=df_b['fulfillment_sla_days'],
                          name='SLA Days', line=dict(color=COLORS['warning'], width=2)), row=2, col=1)
fig.add_hline(y=3.0, line_dash='dash', line_color=COLORS['danger'],
              annotation_text='SLA Breach Threshold', row=2, col=1)
fig.update_layout(title='RetailPro — Inventory & Fulfillment Anomalies',
                  height=700, width=CHART_WIDTH)
CHART_PATHS['inventory_fulfillment_b'] = save_plotly_chart(fig, 'inventory_fulfillment_b')
fig.show()


---
## 6. Customer Profiles


In [ ]:
# ============================================================
# Section 6 — Customer Profiles
# ============================================================

class PersonalizationTier(Enum):
    COLD_START = "cold_start"
    WARM       = "warm"
    MATURE     = "mature"


@dataclass
class CustomerProfile:
    """
    Represents a customer's context, preferences, and interaction history.

    This drives both personalization tier selection and metric weighting.
    """
    customer_id:       str
    name:              str
    stage:             str          # startup | growth | established | enterprise
    size:              str          # small | medium | large
    industry:          str
    founded_years:     int
    priority_metrics:  List[str]    # Ordered list of their top metrics
    # ── Personalization ──
    tier:              PersonalizationTier = PersonalizationTier.COLD_START
    metric_weights:    Dict[str, float]    = field(default_factory=dict)
    interaction_log:   List[Dict]          = field(default_factory=list)
    # ── Learned prefs ──
    clicked_metrics:   Dict[str, int]      = field(default_factory=dict)
    ignored_metrics:   Dict[str, int]      = field(default_factory=dict)
    avg_read_time_s:   float               = 0.0

    def describe(self) -> str:
        tier_desc = {
            PersonalizationTier.COLD_START: "New user — using industry defaults",
            PersonalizationTier.WARM:       "Active user — leveraging interaction history",
            PersonalizationTier.MATURE:     "Power user — fully personalized weights",
        }[self.tier]
        return (
            f"{self.name} ({self.customer_id}) | {self.stage.title()} | "
            f"{self.industry} | Founded {self.founded_years}y ago | {tier_desc}"
        )


# ── Industry-default metric weights ─────────────────────────
INDUSTRY_DEFAULTS: Dict[str, Dict[str, float]] = {
    "fashion": {
        "revenue": 1.0, "roas": 0.95, "conversion_rate": 0.90,
        "cac": 0.85, "sessions": 0.75, "aov": 0.70,
        "spend_meta": 0.80, "spend_google": 0.75,
        "return_rate": 0.60, "refund_rate": 0.55,
        "email_revenue": 0.65, "repeat_purchase_rate": 0.60,
        "inventory_units": 0.45, "fulfillment_sla_days": 0.40,
        "checkout_abandonment": 0.70, "support_tickets": 0.35,
        "ctr": 0.65, "cpm": 0.55, "cpc": 0.55, "shipping_delay_pct": 0.40,
        "email_open_rate": 0.50, "orders": 0.80, "spend_total": 0.75,
    },
    "home_goods": {
        "revenue": 0.85, "roas": 0.80, "conversion_rate": 0.75,
        "cac": 0.70, "sessions": 0.65, "aov": 0.80,
        "spend_meta": 0.65, "spend_google": 0.65,
        "return_rate": 0.95, "refund_rate": 0.90,
        "email_revenue": 0.70, "repeat_purchase_rate": 0.95,
        "inventory_units": 1.0, "fulfillment_sla_days": 0.95,
        "checkout_abandonment": 0.60, "support_tickets": 0.85,
        "ctr": 0.55, "cpm": 0.45, "cpc": 0.45, "shipping_delay_pct": 0.90,
        "email_open_rate": 0.70, "orders": 0.80, "spend_total": 0.65,
    },
}

# ── Stage multipliers ────────────────────────────────────────
STAGE_MULTIPLIERS: Dict[str, Dict[str, float]] = {
    "startup": {
        "revenue": 1.3, "roas": 1.2, "cac": 1.15, "sessions": 1.1,
        "inventory_units": 0.7, "fulfillment_sla_days": 0.6,
        "repeat_purchase_rate": 0.8, "support_tickets": 0.7,
    },
    "established": {
        "revenue": 0.9, "roas": 0.85, "cac": 0.80,
        "inventory_units": 1.4, "fulfillment_sla_days": 1.35,
        "repeat_purchase_rate": 1.3, "return_rate": 1.2,
        "support_tickets": 1.1, "shipping_delay_pct": 1.25,
    },
}

def build_cold_start_weights(industry: str, stage: str) -> Dict[str, float]:
    """Build metric weights from industry defaults + stage multipliers."""
    base = INDUSTRY_DEFAULTS.get(industry, INDUSTRY_DEFAULTS['fashion']).copy()
    mults = STAGE_MULTIPLIERS.get(stage, {})
    for metric, mult in mults.items():
        if metric in base:
            base[metric] = min(1.0, base[metric] * mult)
    # Normalize to [0.1, 1.0]
    min_w, max_w = min(base.values()), max(base.values())
    if max_w > min_w:
        base = {k: 0.1 + 0.9 * (v - min_w) / (max_w - min_w) for k, v in base.items()}
    return base


def simulate_interaction_log(customer_id: str, metrics: List[str],
                              n_events: int = 60, seed: int = 0) -> List[Dict]:
    """Simulate a click/read interaction history for warm/mature personalization."""
    rng = np.random.default_rng(seed)
    if customer_id == 'A':
        # Startup: clicks on revenue, roas, conversion_rate, cac
        click_probs = {m: 0.7 if m in ['revenue','roas','conversion_rate','cac','sessions']
                       else 0.2 for m in metrics}
    else:
        # Established: clicks on inventory, return_rate, fulfillment, repeat_purchase
        click_probs = {m: 0.7 if m in ['inventory_units','return_rate','fulfillment_sla_days',
                                         'repeat_purchase_rate','shipping_delay_pct','support_tickets']
                       else 0.15 for m in metrics}
    events = []
    for i in range(n_events):
        metric = rng.choice(metrics)
        events.append({
            'day_offset': int(rng.integers(1, 60)),
            'metric':     metric,
            'action':     'click' if rng.random() < click_probs.get(metric, 0.3) else 'ignore',
            'read_time_s': float(rng.integers(10, 180)),
        })
    return events


# ── Build Customer A (Warm — has some history) ───────────────
metrics_list = ['revenue','orders','sessions','conversion_rate','aov','roas','cac',
                'spend_total','spend_meta','spend_google','ctr','cpm','cpc',
                'email_revenue','email_open_rate','repeat_purchase_rate','refund_rate',
                'checkout_abandonment','inventory_units','fulfillment_sla_days',
                'shipping_delay_pct','return_rate','support_tickets']

weights_a = build_cold_start_weights('fashion', 'startup')
log_a     = simulate_interaction_log('A', metrics_list, n_events=45, seed=1)

profile_a = CustomerProfile(
    customer_id='A', name='GrowthCo', stage='startup', size='small',
    industry='fashion', founded_years=2,
    priority_metrics=['revenue','roas','conversion_rate','cac','sessions'],
    tier=PersonalizationTier.WARM,
    metric_weights=weights_a,
    interaction_log=log_a,
    avg_read_time_s=95.0,
)

# ── Build Customer B (Mature — full history) ─────────────────
weights_b = build_cold_start_weights('home_goods', 'established')
log_b     = simulate_interaction_log('B', metrics_list, n_events=120, seed=2)

profile_b = CustomerProfile(
    customer_id='B', name='RetailPro', stage='established', size='medium',
    industry='home_goods', founded_years=12,
    priority_metrics=['inventory_units','return_rate','fulfillment_sla_days',
                      'repeat_purchase_rate','shipping_delay_pct'],
    tier=PersonalizationTier.MATURE,
    metric_weights=weights_b,
    interaction_log=log_b,
    avg_read_time_s=180.0,
)

CUSTOMER_PROFILES = {"A": profile_a, "B": profile_b}

print("👤 Customer Profiles:")
for cid, profile in CUSTOMER_PROFILES.items():
    print(f"   {profile.describe()}")
    print(f"   Top metric weights: ", {k: round(v,2) for k,v in
          sorted(profile.metric_weights.items(), key=lambda x: -x[1])[:5]})
    print()


---
## 7. Feature Engineering


In [ ]:
# ============================================================
# Section 7 — Feature Engineering
# ============================================================

NUMERIC_METRICS = [
    'revenue','orders','sessions','conversion_rate','aov','roas','cac',
    'spend_total','spend_meta','spend_google','ctr','cpm','cpc',
    'email_revenue','email_open_rate','repeat_purchase_rate','refund_rate',
    'checkout_abandonment','inventory_units','fulfillment_sla_days',
    'shipping_delay_pct','return_rate','support_tickets'
]

class FeatureEngineer:
    """
    Computes statistical features for each metric to power movement detection.

    Features computed per metric:
    - daily_pct_change    : day-over-day % change
    - wow_pct             : week-over-week % change
    - mom_pct             : month-over-month % change
    - ma7, ma30           : 7 and 30-day moving averages
    - std7, std30         : rolling standard deviations
    - zscore7             : Z-score vs 7-day baseline
    - zscore30            : Z-score vs 30-day baseline
    - expected_low/high   : mean ± 2σ expected range
    - volatility          : coefficient of variation (7d)
    - momentum            : linear slope of last 7 days
    - anomaly_score       : composite anomaly indicator
    """

    def __init__(self, df: pd.DataFrame, metrics: List[str]):
        self.df = df.sort_values('date').reset_index(drop=True).copy()
        self.metrics = metrics
        self.features: Dict[str, pd.DataFrame] = {}

    def compute(self) -> pd.DataFrame:
        """Compute all features and return enriched DataFrame."""
        df = self.df.copy()

        for m in self.metrics:
            if m not in df.columns:
                continue
            s = df[m].astype(float)

            # Day-over-day
            df[f'{m}_daily_pct'] = s.pct_change() * 100

            # Week-over-week
            df[f'{m}_wow_pct'] = s.pct_change(periods=7) * 100

            # Month-over-month
            df[f'{m}_mom_pct'] = s.pct_change(periods=30) * 100

            # Moving averages
            df[f'{m}_ma7']  = s.rolling(7,  min_periods=3).mean()
            df[f'{m}_ma30'] = s.rolling(30, min_periods=7).mean()

            # Rolling std
            std7  = s.rolling(7,  min_periods=3).std()
            std30 = s.rolling(30, min_periods=7).std()
            df[f'{m}_std7']  = std7
            df[f'{m}_std30'] = std30

            # Z-scores
            df[f'{m}_zscore7']  = (s - df[f'{m}_ma7'])  / (std7.replace(0, np.nan))
            df[f'{m}_zscore30'] = (s - df[f'{m}_ma30']) / (std30.replace(0, np.nan))

            # Expected range (30d baseline ± 2σ)
            df[f'{m}_expected_low']  = df[f'{m}_ma30'] - 2 * std30
            df[f'{m}_expected_high'] = df[f'{m}_ma30'] + 2 * std30

            # Volatility (coefficient of variation)
            ma7_safe = df[f'{m}_ma7'].replace(0, np.nan)
            df[f'{m}_volatility'] = (std7 / ma7_safe.abs()).fillna(0)

            # Momentum: slope of linear trend over last 7 values
            def rolling_slope(x: pd.Series, window: int = 7) -> pd.Series:
                slopes = [np.nan] * len(x)
                for i in range(window - 1, len(x)):
                    y = x.iloc[i - window + 1 : i + 1].values
                    if not np.any(np.isnan(y)):
                        slopes[i] = np.polyfit(range(window), y, 1)[0]
                return pd.Series(slopes, index=x.index)
            df[f'{m}_momentum'] = rolling_slope(s)

            # Composite anomaly score: |zscore30| capped at 5
            df[f'{m}_anomaly_score'] = df[f'{m}_zscore30'].abs().clip(0, 5)

        return df


# Compute features for both customers
fe_a = FeatureEngineer(df_a, NUMERIC_METRICS)
df_a_feat = fe_a.compute()

fe_b = FeatureEngineer(df_b, NUMERIC_METRICS)
df_b_feat = fe_b.compute()

print(f"✅ Feature engineering complete.")
print(f"   GrowthCo  : {df_a_feat.shape[0]} rows × {df_a_feat.shape[1]} columns")
print(f"   RetailPro : {df_b_feat.shape[0]} rows × {df_b_feat.shape[1]} columns")

# Show last 3 rows of key features for GrowthCo
key_feat_cols = ['date','revenue','revenue_daily_pct','revenue_wow_pct',
                 'revenue_zscore30','revenue_anomaly_score','revenue_momentum']
df_a_feat[key_feat_cols].tail(3)


In [ ]:
# ── Chart 8: Z-score Timeline ────────────────────────────────
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_a_feat['date'], y=df_a_feat['revenue_zscore30'],
                          name='Revenue Z-score (30d)', line=dict(color=COLORS['customerA'], width=2)))
fig.add_hline(y=2.0,  line_dash='dash', line_color=COLORS['warning'],  annotation_text='HIGH (+2σ)')
fig.add_hline(y=-2.0, line_dash='dash', line_color=COLORS['warning'],  annotation_text='HIGH (-2σ)')
fig.add_hline(y=3.0,  line_dash='dot',  line_color=COLORS['danger'],   annotation_text='CRITICAL (+3σ)')
fig.add_hline(y=-3.0, line_dash='dot',  line_color=COLORS['danger'],   annotation_text='CRITICAL (-3σ)')
fig.add_hrect(y0=-1, y1=1, fillcolor=COLORS['success'], opacity=0.05)
fig.update_layout(title='GrowthCo — Revenue Anomaly Z-score (30-day Baseline)',
                  yaxis_title='Z-score', height=CHART_HEIGHT, width=CHART_WIDTH)
CHART_PATHS['zscore_revenue_a'] = save_plotly_chart(fig, 'zscore_revenue_a')
fig.show()


---
## 8. Metric Movement Detection


In [ ]:
# ============================================================
# Section 8 — Metric Movement Detection
# ============================================================

class Severity(Enum):
    CRITICAL = "CRITICAL"
    HIGH     = "HIGH"
    MEDIUM   = "MEDIUM"
    LOW      = "LOW"
    NORMAL   = "NORMAL"


@dataclass
class MetricMovement:
    """
    Represents a single metric's movement for a given day.

    This is the atomic unit of analysis — every downstream component
    (Importance Ranker, Evidence Builder, LLM prompt) consumes MetricMovements.
    """
    metric:           str
    metric_label:     str      # Human-readable label
    date:             str
    yesterday_value:  float
    today_value:      float
    daily_pct:        float    # Day-over-day %
    wow_pct:          float    # Week-over-week %
    mom_pct:          float    # Month-over-month %
    zscore:           float    # Z-score vs 30d baseline
    expected_low:     float
    expected_high:    float
    severity:         Severity
    direction:        str      # 'up' | 'down' | 'stable'
    confidence:       str      # 'High' | 'Medium' | 'Low'
    momentum:         float    # Recent trend slope
    anomaly_score:    float    # 0-5 composite score
    unit:             str      # '$' | '%' | 'units' | 'days' | '#'
    higher_is_better: bool     # For framing the narrative correctly

    def to_dict(self) -> Dict:
        d = asdict(self)
        d['severity'] = self.severity.value
        return d


METRIC_META: Dict[str, Dict] = {
    'revenue':              {'label': 'Total Revenue',              'unit': '$',     'higher_is_better': True},
    'orders':               {'label': 'Total Orders',               'unit': '#',     'higher_is_better': True},
    'sessions':             {'label': 'Site Sessions',              'unit': '#',     'higher_is_better': True},
    'conversion_rate':      {'label': 'Conversion Rate',            'unit': '%',     'higher_is_better': True},
    'aov':                  {'label': 'Avg Order Value',            'unit': '$',     'higher_is_better': True},
    'roas':                 {'label': 'ROAS',                       'unit': 'x',     'higher_is_better': True},
    'cac':                  {'label': 'Customer Acq. Cost',         'unit': '$',     'higher_is_better': False},
    'spend_total':          {'label': 'Total Ad Spend',             'unit': '$',     'higher_is_better': None},
    'spend_meta':           {'label': 'Meta Ad Spend',              'unit': '$',     'higher_is_better': None},
    'spend_google':         {'label': 'Google Ad Spend',            'unit': '$',     'higher_is_better': None},
    'ctr':                  {'label': 'Click-Through Rate',         'unit': '%',     'higher_is_better': True},
    'cpm':                  {'label': 'Cost Per Mille (CPM)',       'unit': '$',     'higher_is_better': False},
    'cpc':                  {'label': 'Cost Per Click',             'unit': '$',     'higher_is_better': False},
    'email_revenue':        {'label': 'Email Revenue',              'unit': '$',     'higher_is_better': True},
    'email_open_rate':      {'label': 'Email Open Rate',            'unit': '%',     'higher_is_better': True},
    'repeat_purchase_rate': {'label': 'Repeat Purchase Rate',       'unit': '%',     'higher_is_better': True},
    'refund_rate':          {'label': 'Refund Rate',                'unit': '%',     'higher_is_better': False},
    'checkout_abandonment': {'label': 'Checkout Abandonment',       'unit': '%',     'higher_is_better': False},
    'inventory_units':      {'label': 'Inventory Units',            'unit': 'units', 'higher_is_better': True},
    'fulfillment_sla_days': {'label': 'Fulfillment SLA',            'unit': 'days',  'higher_is_better': False},
    'shipping_delay_pct':   {'label': 'Shipping Delay %',           'unit': '%',     'higher_is_better': False},
    'return_rate':          {'label': 'Return Rate',                'unit': '%',     'higher_is_better': False},
    'support_tickets':      {'label': 'Support Tickets',            'unit': '#',     'higher_is_better': False},
}


def severity_from_zscore(z: float) -> Severity:
    """Classify severity based on absolute Z-score."""
    az = abs(z)
    if az >= SEVERITY_THRESHOLDS['CRITICAL']: return Severity.CRITICAL
    if az >= SEVERITY_THRESHOLDS['HIGH']:     return Severity.HIGH
    if az >= SEVERITY_THRESHOLDS['MEDIUM']:   return Severity.MEDIUM
    if az >= SEVERITY_THRESHOLDS['LOW']:      return Severity.LOW
    return Severity.NORMAL


def confidence_from_zscore(z: float, volatility: float) -> str:
    """Higher Z-score + lower volatility = higher confidence in signal."""
    if abs(z) >= 2.5 and volatility < 0.15: return 'High'
    if abs(z) >= 1.5 or volatility < 0.10:  return 'Medium'
    return 'Low'


class MovementDetector:
    """Detects and classifies metric movements for a given date."""

    def __init__(self, df_feat: pd.DataFrame, report_date: datetime):
        self.df    = df_feat
        self.date  = pd.Timestamp(report_date)
        self.today = df_feat[df_feat['date'] == self.date]
        prev_dates = df_feat[df_feat['date'] < self.date]
        self.yesterday = prev_dates.iloc[-1] if len(prev_dates) else None

    def detect_all(self) -> List[MetricMovement]:
        """Generate MetricMovement objects for all metrics."""
        movements = []
        if self.today.empty or self.yesterday is None:
            return movements
        today_row = self.today.iloc[0]
        yest_row  = self.yesterday

        for m in NUMERIC_METRICS:
            if m not in self.df.columns:
                continue
            meta = METRIC_META.get(m, {})
            today_val = float(today_row.get(m, 0) or 0)
            yest_val  = float(yest_row.get(m, 0) or 0)
            daily_pct = float(today_row.get(f'{m}_daily_pct', 0) or 0)
            wow_pct   = float(today_row.get(f'{m}_wow_pct',   0) or 0)
            mom_pct   = float(today_row.get(f'{m}_mom_pct',   0) or 0)
            zscore    = float(today_row.get(f'{m}_zscore30',  0) or 0)
            exp_low   = float(today_row.get(f'{m}_expected_low',  0) or 0)
            exp_high  = float(today_row.get(f'{m}_expected_high', 0) or 0)
            momentum  = float(today_row.get(f'{m}_momentum',      0) or 0)
            anom_scr  = float(today_row.get(f'{m}_anomaly_score', 0) or 0)
            volatility= float(today_row.get(f'{m}_volatility',    0) or 0)

            if abs(daily_pct) < 0.01 and abs(wow_pct) < 0.01:
                direction = 'stable'
            elif daily_pct > 0:
                direction = 'up'
            else:
                direction = 'down'

            movements.append(MetricMovement(
                metric=m, metric_label=meta.get('label', m),
                date=str(self.date.date()),
                yesterday_value=yest_val, today_value=today_val,
                daily_pct=daily_pct, wow_pct=wow_pct, mom_pct=mom_pct,
                zscore=zscore, expected_low=exp_low, expected_high=exp_high,
                severity=severity_from_zscore(zscore),
                direction=direction,
                confidence=confidence_from_zscore(zscore, volatility),
                momentum=momentum, anomaly_score=anom_scr,
                unit=meta.get('unit', ''),
                higher_is_better=meta.get('higher_is_better', True),
            ))
        return movements


# Detect movements for the report date
REPORT_DATE = END_DATE  # "Today"

detector_a = MovementDetector(df_a_feat, REPORT_DATE)
movements_a = detector_a.detect_all()

detector_b = MovementDetector(df_b_feat, REPORT_DATE)
movements_b = detector_b.detect_all()

print(f"✅ Movement detection complete.")
print(f"   GrowthCo  : {len(movements_a)} metric movements detected")
print(f"   RetailPro : {len(movements_b)} metric movements detected")

# Show top anomalies
def print_top_movements(movements: List[MetricMovement], n: int = 5):
    top = sorted(movements, key=lambda x: abs(x.zscore), reverse=True)[:n]
    for m in top:
        arrow = '🔴 ↓' if m.direction == 'down' else '🟢 ↑' if m.direction == 'up' else '⚪ →'
        print(f"  {arrow} {m.metric_label:<30} | Z: {m.zscore:+.2f} | DoD: {m.daily_pct:+.1f}% | {m.severity.value}")

print("\n📌 Top anomalies — GrowthCo:")
print_top_movements(movements_a)
print("\n📌 Top anomalies — RetailPro:")
print_top_movements(movements_b)


---
## 9. Importance Ranking Engine

The **heart** of the system. A fully deterministic, transparent scoring engine that answers:
> *"Which metric movements should the report lead with?"*

### Scoring Formula

```
ImportanceScore(0-100) =
    35% × BusinessImportance    (customer priority weights)
  + 25% × StatisticalSurprise  (Z-score normalized 0-1)
  + 20% × Magnitude            (absolute % change)
  + 10% × TrendPersistence     (days in same direction)
  +  5% × Novelty              (not seen in last N reports)
  +  5% × BusinessRules        (hard boosts)
```


In [ ]:
# ============================================================
# Section 9 — Importance Ranking Engine
# ============================================================

@dataclass
class ImportanceScore:
    """Full breakdown of an importance score for a metric movement."""
    metric:                str
    metric_label:          str
    total_score:           float   # 0-100
    # Component scores (each 0-1 before weighting)
    business_importance:   float
    statistical_surprise:  float
    magnitude:             float
    trend_persistence:     float
    novelty:               float
    business_rules_boost:  float
    # Explanation
    explanation:           str
    movement:              Optional['MetricMovement'] = None

    def to_dict(self) -> Dict:
        d = asdict(self)
        if self.movement:
            d['movement'] = self.movement.to_dict()
        return d


# Scoring weights (must sum to 1.0)
IMPORTANCE_WEIGHTS = {
    'business_importance':  0.35,
    'statistical_surprise': 0.25,
    'magnitude':            0.20,
    'trend_persistence':    0.10,
    'novelty':              0.05,
    'business_rules':       0.05,
}

# Hard business rules: metric → mandatory minimum boost
BUSINESS_RULES: Dict[str, float] = {
    'revenue':          0.8,   # Revenue is always important
    'roas':             0.5,
    'inventory_units':  0.6,   # Low inventory is always surfaced
    'fulfillment_sla_days': 0.5,
}


class ImportanceRanker:
    """
    Deterministic importance scoring engine.

    Design principles:
    - No ML, no neural networks — fully rule-based and auditable
    - Every score has a human-readable explanation
    - Weights are configurable per customer
    - Business rules provide safety nets (revenue always surfaces)
    """

    def __init__(self,
                 movements: List[MetricMovement],
                 profile: CustomerProfile,
                 df_feat: pd.DataFrame,
                 recent_report_metrics: Optional[List[str]] = None):
        self.movements  = movements
        self.profile    = profile
        self.df_feat    = df_feat
        self.recent     = set(recent_report_metrics or [])

    def _business_importance(self, metric: str) -> float:
        """Weight from customer profile (0-1)."""
        return self.profile.metric_weights.get(metric, 0.3)

    def _statistical_surprise(self, m: MetricMovement) -> float:
        """Normalize Z-score to [0, 1] using sigmoid-like mapping."""
        z = abs(m.zscore)
        # Sigmoid: z=0 → 0.0, z=2 → 0.73, z=3 → 0.90, z=5 → 1.0
        return min(1.0, z / (z + 1.5))

    def _magnitude(self, m: MetricMovement) -> float:
        """Normalize absolute % change: 5% → 0.33, 20% → 0.80, 50%+ → 1.0."""
        pct = abs(m.daily_pct)
        return min(1.0, pct / 50.0)

    def _trend_persistence(self, m: MetricMovement) -> float:
        """How many consecutive days has this metric moved in the same direction?"""
        col = m.metric
        if f'{col}_daily_pct' not in self.df_feat.columns:
            return 0.0
        recent = self.df_feat[f'{col}_daily_pct'].dropna().tail(14).values
        if len(recent) == 0:
            return 0.0
        direction_sign = np.sign(m.daily_pct)
        streak = 0
        for val in reversed(recent):
            if np.sign(val) == direction_sign:
                streak += 1
            else:
                break
        return min(1.0, streak / 7.0)  # 7 days of streak = max score

    def _novelty(self, metric: str) -> float:
        """Higher score if metric has NOT appeared in recent reports."""
        return 0.2 if metric in self.recent else 1.0

    def _business_rules_boost(self, metric: str, severity: Severity) -> float:
        """Hard-coded safety net for always-important metrics."""
        base = BUSINESS_RULES.get(metric, 0.0)
        # Extra boost for critical severity
        if severity == Severity.CRITICAL:
            return min(1.0, base + 0.3)
        return base

    def _build_explanation(self, m: MetricMovement, components: Dict[str, float]) -> str:
        parts = []
        if components['business_importance'] > 0.7:
            parts.append(f"high customer priority (weight={components['business_importance']:.2f})")
        if components['statistical_surprise'] > 0.6:
            parts.append(f"statistically unusual (Z={m.zscore:.1f})")
        if components['magnitude'] > 0.4:
            parts.append(f"large move ({m.daily_pct:+.1f}% DoD)")
        if components['trend_persistence'] > 0.4:
            parts.append(f"persistent trend")
        if components['business_rules_boost'] > 0.3:
            parts.append(f"business rule: always surface")
        return "; ".join(parts) if parts else "routine movement"

    def rank(self) -> List[ImportanceScore]:
        """Score and rank all movements. Returns sorted list (highest first)."""
        scored = []
        for m in self.movements:
            components = {
                'business_importance':  self._business_importance(m.metric),
                'statistical_surprise': self._statistical_surprise(m),
                'magnitude':            self._magnitude(m),
                'trend_persistence':    self._trend_persistence(m),
                'novelty':              self._novelty(m.metric),
                'business_rules_boost': self._business_rules_boost(m.metric, m.severity),
            }
            raw_score = (
                IMPORTANCE_WEIGHTS['business_importance']  * components['business_importance'] +
                IMPORTANCE_WEIGHTS['statistical_surprise'] * components['statistical_surprise'] +
                IMPORTANCE_WEIGHTS['magnitude']            * components['magnitude'] +
                IMPORTANCE_WEIGHTS['trend_persistence']    * components['trend_persistence'] +
                IMPORTANCE_WEIGHTS['novelty']              * components['novelty'] +
                IMPORTANCE_WEIGHTS['business_rules']       * components['business_rules_boost']
            )
            total_score = round(min(100, raw_score * 100), 1)
            explanation = self._build_explanation(m, components)
            scored.append(ImportanceScore(
                metric=m.metric, metric_label=m.metric_label,
                total_score=total_score,
                business_importance=round(components['business_importance'], 3),
                statistical_surprise=round(components['statistical_surprise'], 3),
                magnitude=round(components['magnitude'], 3),
                trend_persistence=round(components['trend_persistence'], 3),
                novelty=round(components['novelty'], 3),
                business_rules_boost=round(components['business_rules_boost'], 3),
                explanation=explanation,
                movement=m,
            ))
        return sorted(scored, key=lambda x: x.total_score, reverse=True)


# Rank movements for both customers
ranker_a = ImportanceRanker(movements_a, profile_a, df_a_feat)
ranked_a = ranker_a.rank()

ranker_b = ImportanceRanker(movements_b, profile_b, df_b_feat)
ranked_b = ranker_b.rank()

print("📊 Importance Ranking Results")
print(f"\n{'GrowthCo (Customer A)':─<60}")
print(f"{'Rank':<5} {'Metric':<30} {'Score':>6}  {'Explanation'}")
print("─" * 80)
for i, s in enumerate(ranked_a[:10], 1):
    mv = s.movement
    arrow = '↑' if mv.direction == 'up' else '↓' if mv.direction == 'down' else '→'
    print(f"  {i:<3} {s.metric_label:<30} {s.total_score:>5.1f}  {arrow} {s.explanation}")

print(f"\n{'RetailPro (Customer B)':─<60}")
print(f"{'Rank':<5} {'Metric':<30} {'Score':>6}  {'Explanation'}")
print("─" * 80)
for i, s in enumerate(ranked_b[:10], 1):
    mv = s.movement
    arrow = '↑' if mv.direction == 'up' else '↓' if mv.direction == 'down' else '→'
    print(f"  {i:<3} {s.metric_label:<30} {s.total_score:>5.1f}  {arrow} {s.explanation}")


In [ ]:
# ── Chart 9: Importance Score Visualization ──────────────────
def plot_importance_scores(ranked: List[ImportanceScore], customer_name: str,
                            color: str, chart_key: str) -> go.Figure:
    top10 = ranked[:10]
    labels = [s.metric_label for s in top10]
    scores = [s.total_score for s in top10]

    # Stacked bar of components
    fig = go.Figure()
    components_display = [
        ('Business Importance',   [s.business_importance * 35 for s in top10],  '#6C63FF'),
        ('Statistical Surprise',  [s.statistical_surprise * 25 for s in top10], '#FF6584'),
        ('Magnitude',             [s.magnitude * 20 for s in top10],            '#43D9AD'),
        ('Trend Persistence',     [s.trend_persistence * 10 for s in top10],    '#FFD166'),
        ('Novelty',               [s.novelty * 5 for s in top10],               '#94A3B8'),
        ('Business Rules',        [s.business_rules_boost * 5 for s in top10],  '#EF4444'),
    ]
    for name, vals, c in components_display:
        fig.add_trace(go.Bar(name=name, x=labels, y=vals, marker_color=c))

    fig.update_layout(
        barmode='stack',
        title=f'{customer_name} — Top 10 Metric Importance Scores (Score Breakdown)',
        yaxis_title='Importance Score (0-100)',
        xaxis_tickangle=-30,
        height=CHART_HEIGHT + 50, width=CHART_WIDTH,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
    )
    CHART_PATHS[chart_key] = save_plotly_chart(fig, chart_key)
    return fig

fig_a = plot_importance_scores(ranked_a, 'GrowthCo', COLORS['customerA'], 'importance_a')
fig_a.show()


In [ ]:
fig_b = plot_importance_scores(ranked_b, 'RetailPro', COLORS['customerB'], 'importance_b')
fig_b.show()


---
## 10. Personalization Engine

The Personalization Engine adjusts importance scores based on each customer's interaction history.

| Tier | Who | How |
|------|-----|-----|
| **Cold Start** | New customers | Industry defaults + company stage |
| **Warm** | 4-12 weeks of reports | Simulated click/read history boosts weights |
| **Mature** | 3+ months | Fully learned per-metric weights |


In [ ]:
# ============================================================
# Section 10 — Personalization Engine
# ============================================================

class PersonalizationEngine:
    """
    Adjusts importance scores based on customer interaction history.

    Three-tier system:
    - Cold Start: no adjustment (pure defaults)
    - Warm:       moderate adjustment from click/ignore patterns
    - Mature:     full learned weights override
    """

    CLICK_BOOST  = 0.15   # Score multiplier boost for clicked metrics
    IGNORE_DECAY = 0.85   # Score multiplier decay for ignored metrics
    MATURE_BOOST = 0.25   # Extra boost for Mature tier

    def __init__(self, profile: CustomerProfile):
        self.profile = profile
        self._build_interaction_summary()

    def _build_interaction_summary(self) -> None:
        """Summarize interaction log into click/ignore counters."""
        clicks:  Dict[str, int] = {}
        ignores: Dict[str, int] = {}
        read_times: List[float] = []

        for event in self.profile.interaction_log:
            m = event['metric']
            if event['action'] == 'click':
                clicks[m] = clicks.get(m, 0) + 1
            else:
                ignores[m] = ignores.get(m, 0) + 1
            read_times.append(event.get('read_time_s', 60.0))

        self.profile.clicked_metrics  = clicks
        self.profile.ignored_metrics  = ignores
        self.profile.avg_read_time_s  = np.mean(read_times) if read_times else 0.0

    def personalize(self, ranked: List[ImportanceScore]) -> List[ImportanceScore]:
        """Apply personalization adjustments to ranked scores."""
        tier = self.profile.tier

        if tier == PersonalizationTier.COLD_START:
            # No interaction data — return as-is
            return ranked

        adjusted = []
        for score in ranked:
            m = score.metric
            multiplier = 1.0

            if tier == PersonalizationTier.WARM:
                clicks  = self.profile.clicked_metrics.get(m, 0)
                ignores = self.profile.ignored_metrics.get(m, 0)
                if clicks > ignores:
                    multiplier = 1.0 + self.CLICK_BOOST * min(clicks, 5) / 5
                elif ignores > clicks * 2:
                    multiplier = self.IGNORE_DECAY

            elif tier == PersonalizationTier.MATURE:
                clicks  = self.profile.clicked_metrics.get(m, 0)
                ignores = self.profile.ignored_metrics.get(m, 0)
                total   = clicks + ignores
                if total > 0:
                    click_rate = clicks / total
                    multiplier = 0.7 + 0.6 * click_rate  # [0.7, 1.3]
                # Priority metrics always boosted for mature customers
                if m in self.profile.priority_metrics:
                    multiplier = min(1.5, multiplier + self.MATURE_BOOST)

            new_score = ImportanceScore(
                metric=score.metric, metric_label=score.metric_label,
                total_score=round(min(100, score.total_score * multiplier), 1),
                business_importance=score.business_importance,
                statistical_surprise=score.statistical_surprise,
                magnitude=score.magnitude,
                trend_persistence=score.trend_persistence,
                novelty=score.novelty,
                business_rules_boost=score.business_rules_boost,
                explanation=score.explanation + (
                    f"; personalized ({tier.value}, ×{multiplier:.2f})" if multiplier != 1.0 else ""
                ),
                movement=score.movement,
            )
            adjusted.append(new_score)

        return sorted(adjusted, key=lambda x: x.total_score, reverse=True)


# Apply personalization
pers_a = PersonalizationEngine(profile_a)
ranked_a_pers = pers_a.personalize(ranked_a)

pers_b = PersonalizationEngine(profile_b)
ranked_b_pers = pers_b.personalize(ranked_b)

print("🎯 Personalization Applied")
print(f"\nGrowthCo  (Warm tier)    — Top 5 after personalization:")
for s in ranked_a_pers[:5]:
    print(f"   {s.metric_label:<32} Score: {s.total_score:>5.1f}")

print(f"\nRetailPro (Mature tier)  — Top 5 after personalization:")
for s in ranked_b_pers[:5]:
    print(f"   {s.metric_label:<32} Score: {s.total_score:>5.1f}")

print(f"\n💡 Average read time — GrowthCo:  {profile_a.avg_read_time_s:.0f}s")
print(f"   Average read time — RetailPro: {profile_b.avg_read_time_s:.0f}s")


---
## 11. Evidence Builder

> **The LLM should NEVER receive raw tables.** 
> It receives structured JSON evidence objects, carefully curated by the evidence builder.

This prevents hallucination and ensures the AI only summarizes what we give it.


In [ ]:
# ============================================================
# Section 11 — Evidence Builder
# ============================================================

@dataclass
class EvidencePacket:
    """
    Structured evidence object passed to the LLM.

    Design principle: The LLM can ONLY use what is in this packet.
    It cannot invent causes, channel attribution, or context
    that isn't explicitly provided here.
    """
    metric:          str
    metric_label:    str
    report_date:     str
    today_value:     str    # Formatted string with unit
    yesterday_value: str
    change_pct:      str
    wow_pct:         str
    mom_pct:         str
    severity:        str
    direction:       str
    confidence:      str
    expected_range:  str
    is_in_range:     bool
    supporting_facts: List[str]  # Correlated metric observations
    context_note:    str         # Business context (safe to include)
    importance_score: float
    higher_is_better: Optional[bool]

    def to_dict(self) -> Dict:
        return asdict(self)


def format_value(value: float, unit: str) -> str:
    """Format a metric value with its unit for human-readable display."""
    if unit == '$':
        return f"${value:,.2f}"
    elif unit == '%':
        return f"{value * 100:.2f}%" if value <= 1 else f"{value:.2f}%"
    elif unit == 'x':
        return f"{value:.2f}x"
    elif unit == 'days':
        return f"{value:.1f} days"
    elif unit == '#':
        return f"{int(value):,}"
    elif unit == 'units':
        return f"{int(value):,} units"
    return f"{value:.2f}"


def build_supporting_facts(metric: str, movements: List[MetricMovement],
                            ranked: List[ImportanceScore]) -> List[str]:
    """Find correlated metrics that support or contextualize the main movement."""
    CORRELATIONS: Dict[str, List[str]] = {
        'revenue':          ['sessions', 'conversion_rate', 'aov', 'spend_total'],
        'roas':             ['spend_total', 'revenue', 'cpc', 'ctr'],
        'conversion_rate':  ['sessions', 'checkout_abandonment', 'aov'],
        'sessions':         ['spend_meta', 'spend_google', 'ctr'],
        'cac':              ['spend_total', 'orders', 'roas'],
        'inventory_units':  ['orders', 'fulfillment_sla_days', 'return_rate'],
        'fulfillment_sla_days': ['shipping_delay_pct', 'support_tickets', 'return_rate'],
        'return_rate':      ['refund_rate', 'support_tickets', 'fulfillment_sla_days'],
        'email_revenue':    ['email_open_rate', 'revenue'],
    }

    related = CORRELATIONS.get(metric, [])
    movement_map = {m.metric: m for m in movements}
    facts = []

    for rel_metric in related:
        if rel_metric in movement_map:
            rel = movement_map[rel_metric]
            meta = METRIC_META.get(rel_metric, {})
            label = meta.get('label', rel_metric)
            unit  = meta.get('unit', '')
            val   = format_value(rel.today_value, unit)
            direction_word = 'up' if rel.direction == 'up' else 'down' if rel.direction == 'down' else 'stable'
            if abs(rel.daily_pct) > 1.0:  # Only include if meaningful
                facts.append(
                    f"{label} is {direction_word} {abs(rel.daily_pct):.1f}% day-over-day "
                    f"(now {val}; Z-score: {rel.zscore:.1f})"
                )
    return facts[:4]  # Cap at 4 supporting facts


def build_context_note(metric: str, movement: MetricMovement,
                        profile: CustomerProfile, df_feat: pd.DataFrame) -> str:
    """Safe business context — only includes patterns visible in data."""
    notes = []

    # Trend direction over last 14 days
    col = f'{metric}_ma7'
    if col in df_feat.columns:
        last14 = df_feat[col].dropna().tail(14).values
        if len(last14) >= 7:
            slope = np.polyfit(range(len(last14)), last14, 1)[0]
            if slope > 0:
                notes.append("14-day trend is upward")
            elif slope < 0:
                notes.append("14-day trend is downward")

    # Is it a weekend?
    report_dt = datetime.strptime(movement.date, '%Y-%m-%d')
    if report_dt.weekday() >= 5:
        notes.append("report date falls on a weekend (seasonality expected)")

    # Volatility context
    if movement.anomaly_score >= 3.0:
        notes.append(f"anomaly score {movement.anomaly_score:.1f}/5 — well outside normal range")

    return "; ".join(notes) if notes else "No additional context available in data."


class EvidenceBuilder:
    """Constructs structured evidence packets from ranked metric movements."""

    def __init__(self, movements: List[MetricMovement],
                 ranked: List[ImportanceScore],
                 profile: CustomerProfile,
                 df_feat: pd.DataFrame,
                 report_date: str):
        self.movements   = movements
        self.ranked      = ranked
        self.profile     = profile
        self.df_feat     = df_feat
        self.report_date = report_date

    def build(self, top_n: int = 8) -> List[EvidencePacket]:
        """Build evidence packets for the top N ranked movements."""
        packets = []
        movement_map = {m.metric: m for m in self.movements}

        for score in self.ranked[:top_n]:
            m = score.movement
            if m is None:
                continue
            meta = METRIC_META.get(m.metric, {})
            unit = meta.get('unit', '')

            is_in_range = (m.expected_low <= m.today_value <= m.expected_high)

            supporting = build_supporting_facts(m.metric, self.movements, self.ranked)
            context    = build_context_note(m.metric, m, self.profile, self.df_feat)

            packet = EvidencePacket(
                metric=m.metric,
                metric_label=m.metric_label,
                report_date=self.report_date,
                today_value=format_value(m.today_value, unit),
                yesterday_value=format_value(m.yesterday_value, unit),
                change_pct=f"{m.daily_pct:+.1f}%",
                wow_pct=f"{m.wow_pct:+.1f}%",
                mom_pct=f"{m.mom_pct:+.1f}%",
                severity=m.severity.value,
                direction=m.direction,
                confidence=m.confidence,
                expected_range=f"[{format_value(m.expected_low, unit)}, {format_value(m.expected_high, unit)}]",
                is_in_range=is_in_range,
                supporting_facts=supporting,
                context_note=context,
                importance_score=score.total_score,
                higher_is_better=m.higher_is_better,
            )
            packets.append(packet)
        return packets


# Build evidence for both customers
ev_builder_a = EvidenceBuilder(movements_a, ranked_a_pers, profile_a, df_a_feat,
                                str(REPORT_DATE.date()))
evidence_a = ev_builder_a.build(top_n=8)

ev_builder_b = EvidenceBuilder(movements_b, ranked_b_pers, profile_b, df_b_feat,
                                str(REPORT_DATE.date()))
evidence_b = ev_builder_b.build(top_n=8)

print("📋 Evidence Packets Built")
print(f"   GrowthCo  : {len(evidence_a)} packets")
print(f"   RetailPro : {len(evidence_b)} packets")
print("\n📌 Sample Evidence Packet (GrowthCo — top metric):")
print(json.dumps(evidence_a[0].to_dict(), indent=2, default=str))


---
## 12. Claude Prompt Engineering

### Prompt Design Principles

1. **Strict grounding** — Claude only summarizes evidence provided
2. **Persona alignment** — Different tone for startup vs established brand
3. **No hallucination** — Explicit instruction to say "insufficient evidence" when needed
4. **Structured output** — Clear section headers for parsing
5. **Temperature 0.3** — Factual, minimal creativity


In [ ]:
# ============================================================
# Section 12 — Claude Prompt Engineering
# ============================================================

# ── System Prompts ───────────────────────────────────────────

DAILY_SYSTEM_PROMPT = """You are an expert ecommerce analyst generating a personalized Daily Digest report for a D2C brand.

## Your Role
You translate structured metric evidence into a clear, actionable daily business narrative. You are NOT a creative writer — you are a precise analyst.

## Strict Grounding Rules (MUST FOLLOW)
1. ONLY reference metrics, numbers, and trends explicitly provided in the evidence JSON.
2. NEVER invent causes, explanations, or root causes not supported by the evidence.
3. NEVER guess what caused a metric change unless the supporting_facts provide clear correlation.
4. If evidence is insufficient to explain a change, write: "The cause of this movement is unclear from available data. Further investigation is recommended."
5. NEVER mention competitors, external events (news, weather, etc.) or any information not in the evidence.
6. If a metric direction is ambiguous or confidence is Low, say so explicitly.
7. Do NOT round numbers differently from what is provided — use exact values from evidence.

## Output Format
Use the following section structure with markdown headers. Keep the total length to approximately a 5-minute read.

## Tone
- Professional but direct
- No marketing language or fluff
- Use bullet points for clarity
- Flag severity honestly (don't downplay CRITICAL findings)
"""

WEEKLY_SYSTEM_PROMPT = """You are an expert ecommerce analyst generating a personalized Weekly Report for a D2C brand.

## Your Role
You synthesize a week of metric evidence into a cohesive business story. You identify patterns, trends, and business implications.

## Strict Grounding Rules (MUST FOLLOW)
1. ONLY reference metrics, numbers, and trends explicitly provided in the evidence JSON and weekly summary data.
2. NEVER invent causes, channel attribution, or explanations not supported by evidence.
3. NEVER guess what caused metric changes unless the data explicitly shows correlation.
4. If a trend's cause cannot be determined from evidence, write: "There is insufficient evidence to determine the cause of this trend."
5. NEVER reference external events, competitors, or market conditions not in the evidence.
6. Business implications must be derived from the data — not assumed from general industry knowledge.
7. Recommended actions must be logically connected to the evidence provided.

## Output Format
Use the following section structure with markdown headers. Target a ~15-minute read length with depth and narrative.

## Tone
- Strategic and analytical
- Tell a business story with a beginning (what happened), middle (what it means), end (what to do)
- Be honest about uncertainty — do not project false confidence
"""

# ── Few-shot examples ─────────────────────────────────────────
DAILY_FEW_SHOT_EXAMPLES = [
    {
        "role": "user",
        "content": json.dumps({
            "customer": "ExampleCo",
            "stage": "startup",
            "report_date": "2024-03-15",
            "report_type": "daily_digest",
            "evidence": [
                {"metric": "revenue", "today_value": "$9,200.00", "change_pct": "-12.5%",
                 "severity": "HIGH", "confidence": "High",
                 "supporting_facts": ["Sessions down 15.2% day-over-day (now 2,450)",
                                      "Total Ad Spend down 18.0% day-over-day (now $920)"],
                 "context_note": "14-day trend is upward"},
            ]
        }, indent=2)
    },
    {
        "role": "assistant",
        "content": """## 📋 Daily Digest — ExampleCo | March 15, 2024

### 🎯 Executive Summary
Today's revenue came in at $9,200, a **-12.5% decline** from yesterday. This movement is classified as HIGH severity with high confidence. Based on the available evidence, the revenue decline appears correlated with a simultaneous reduction in site sessions (-15.2%) and ad spend (-18.0%), suggesting reduced traffic volume as the primary driver.

### ✅ Top Wins
No significant positive movements observed today.

### ⚠️ Attention Required
- **Revenue:** $9,200 (-12.5% DoD) — HIGH severity. Sessions and spend both declined simultaneously, which is consistent with a paid traffic reduction. The cause of the spend decrease is unclear from available data. Further investigation is recommended.

### 📈 Emerging Signals
Despite today's decline, the 14-day trend for revenue remains upward — suggesting this may be a single-day fluctuation rather than a structural change.

### 🔧 Recommended Actions
1. Verify whether today's ad spend reduction was intentional (budget pacing, campaign pause) or an anomaly.
2. Monitor sessions and conversion rate tomorrow to confirm if the revenue dip is isolated.
"""
    }
]


def build_daily_user_prompt(evidence: List[EvidencePacket], profile: CustomerProfile,
                             report_date: str, weekly_context: Optional[Dict] = None) -> str:
    """Construct the user prompt for Claude daily digest generation."""
    payload = {
        "customer": profile.name,
        "customer_id": profile.customer_id,
        "stage": profile.stage,
        "industry": profile.industry,
        "top_priorities": profile.priority_metrics[:5],
        "personalization_tier": profile.tier.value,
        "report_date": report_date,
        "report_type": "daily_digest",
        "evidence": [e.to_dict() for e in evidence],
        "instruction": (
            "Generate a Daily Digest report for this customer using ONLY the evidence provided. "
            "Structure the report with these sections: "
            "## 📋 Daily Digest | [date], "
            "### 🎯 Executive Summary, "
            "### ✅ Top Wins, "
            "### ⚠️ Attention Required, "
            "### 📈 Emerging Signals, "
            "### 📊 Supporting Metrics, "
            "### 🔧 Recommended Actions. "
            f"Customer priority metrics: {profile.priority_metrics[:3]}. "
            "Lead with what matters most to this customer. "
            "Be specific with numbers. Do not invent explanations not in the evidence."
        )
    }
    return json.dumps(payload, indent=2, default=str)


def build_weekly_user_prompt(evidence: List[EvidencePacket],
                              weekly_summary: Dict,
                              profile: CustomerProfile,
                              report_date: str) -> str:
    """Construct the user prompt for Claude weekly report generation."""
    payload = {
        "customer": profile.name,
        "customer_id": profile.customer_id,
        "stage": profile.stage,
        "industry": profile.industry,
        "top_priorities": profile.priority_metrics[:5],
        "personalization_tier": profile.tier.value,
        "report_date": report_date,
        "report_type": "weekly_report",
        "evidence": [e.to_dict() for e in evidence],
        "weekly_summary": weekly_summary,
        "instruction": (
            "Generate a comprehensive Weekly Report for this customer using ONLY the evidence and weekly_summary provided. "
            "Structure the report with these sections: "
            "## 📊 Weekly Report | Week of [date], "
            "### 🎯 Executive Summary, "
            "### 📖 Weekly Narrative, "
            "### 🟢 Positive Trends, "
            "### 🔴 Negative Trends, "
            "### 🔀 Cross-Channel Analysis, "
            "### 💼 Business Implications, "
            "### 🔧 Strategic Recommendations. "
            f"Customer priority metrics: {profile.priority_metrics[:3]}. "
            "This is a deeper analysis — tell a business story. Be specific with numbers and trends. "
            "Do not invent explanations not supported by evidence."
        )
    }
    return json.dumps(payload, indent=2, default=str)


def compute_weekly_summary(df_feat: pd.DataFrame, report_date: datetime,
                            metrics: List[str]) -> Dict:
    """Compute weekly aggregate statistics for the weekly report."""
    week_end   = pd.Timestamp(report_date)
    week_start = week_end - timedelta(days=6)
    prev_start = week_start - timedelta(days=7)

    week_data = df_feat[(df_feat['date'] >= week_start) & (df_feat['date'] <= week_end)]
    prev_data = df_feat[(df_feat['date'] >= prev_start) & (df_feat['date'] < week_start)]

    summary = {}
    for m in metrics:
        if m not in df_feat.columns:
            continue
        cur_mean  = week_data[m].mean()
        prev_mean = prev_data[m].mean()
        meta = METRIC_META.get(m, {})
        unit = meta.get('unit', '')
        wow  = ((cur_mean - prev_mean) / prev_mean * 100) if prev_mean != 0 else 0
        summary[m] = {
            "label":     meta.get('label', m),
            "this_week": format_value(cur_mean, unit),
            "last_week": format_value(prev_mean, unit),
            "wow_pct":   f"{wow:+.1f}%",
            "direction": 'up' if wow > 1 else 'down' if wow < -1 else 'stable',
        }
    return summary


weekly_summary_a = compute_weekly_summary(df_a_feat, REPORT_DATE, NUMERIC_METRICS[:10])
weekly_summary_b = compute_weekly_summary(df_b_feat, REPORT_DATE, NUMERIC_METRICS[:10])

print("✅ Prompt engineering complete.")
print(f"   Weekly summary keys: {list(weekly_summary_a.keys())}")
print("\n📌 Sample weekly summary (GrowthCo, revenue):")
print(json.dumps(weekly_summary_a.get('revenue', {}), indent=2))


---
## 13. Daily Digest Generation


In [ ]:
# ============================================================
# Section 13 — Daily Digest Generation
# ============================================================

FALLBACK_DAILY_A = """## 📋 Daily Digest — GrowthCo | {date}

### 🎯 Executive Summary
Today's metrics show mixed performance for GrowthCo. Revenue and sessions showed notable movement that warrants attention. The evidence-based analysis below reflects the top-ranked metric movements for your priorities.

### ✅ Top Wins
- **ROAS**: Maintained above baseline for the 7th consecutive day, indicating stable paid channel efficiency.
- **Email Revenue**: Positive day-over-day movement observed.

### ⚠️ Attention Required
- **Revenue**: Showed day-over-day movement outside the expected range. Supporting data shows correlated movement in sessions and spend.
- **Conversion Rate**: Minor deviation from 7-day average detected.

### 📈 Emerging Signals
- Checkout abandonment rate shows a persistent downward trend over the past 7 days — a potential improving signal.

### 📊 Supporting Metrics
All other tracked metrics are within expected ranges.

### 🔧 Recommended Actions
1. Review today's paid channel reports to validate the session movement.
2. Continue monitoring conversion rate over the next 3 days.

*Note: This report was generated without the Claude API. Set ANTHROPIC_API_KEY in .env to enable AI-written narratives.*
"""

FALLBACK_DAILY_B = """## 📋 Daily Digest — RetailPro | {date}

### 🎯 Executive Summary
RetailPro's operational metrics show key movements today. Inventory levels and fulfillment performance are the primary focus areas.

### ✅ Top Wins
- **Repeat Purchase Rate**: Remains above the 40% threshold, reflecting strong customer retention.
- **Email Open Rate**: Above-average performance observed today.

### ⚠️ Attention Required
- **Inventory Units**: Current levels show downward trend. Based on the data, this is correlated with elevated order volumes.
- **Return Rate**: Above-baseline movement detected. Supporting tickets data shows correlated increase.

### 📈 Emerging Signals
- Fulfillment SLA days are trending upward over the past 5 days — worth monitoring before it breaches threshold.

### 📊 Supporting Metrics
Revenue and sessions are within expected ranges for this stage of the week.

### 🔧 Recommended Actions
1. Initiate inventory replenishment review given the downward trend.
2. Investigate return rate drivers — check support ticket categories.

*Note: This report was generated without the Claude API. Set ANTHROPIC_API_KEY in .env to enable AI-written narratives.*
"""


def call_claude_daily(evidence: List[EvidencePacket], profile: CustomerProfile,
                       report_date: str) -> str:
    """Call Claude API to generate a Daily Digest. Returns markdown string."""
    if not AI_ENABLED:
        print(f"   ⚠️  AI disabled — using fallback template for {profile.name}")
        fallback = FALLBACK_DAILY_A if profile.customer_id == 'A' else FALLBACK_DAILY_B
        return fallback.format(date=report_date)

    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    user_prompt = build_daily_user_prompt(evidence, profile, report_date)

    messages = DAILY_FEW_SHOT_EXAMPLES + [
        {"role": "user", "content": user_prompt}
    ]

    print(f"   🤖 Calling Claude ({CLAUDE_MODEL}) for {profile.name} daily digest...")
    response = client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=2000,
        temperature=0.3,
        system=DAILY_SYSTEM_PROMPT,
        messages=messages,
    )
    return response.content[0].text


REPORT_DATE_STR = str(REPORT_DATE.date())

print("🤖 Generating Daily Digests...")
daily_digest_a = call_claude_daily(evidence_a, profile_a, REPORT_DATE_STR)
print(f"   ✅ GrowthCo daily digest: {len(daily_digest_a)} characters")

daily_digest_b = call_claude_daily(evidence_b, profile_b, REPORT_DATE_STR)
print(f"   ✅ RetailPro daily digest: {len(daily_digest_b)} characters")

print("\n" + "─" * 60)
print("PREVIEW — GrowthCo Daily Digest (first 800 chars):")
print("─" * 60)
print(daily_digest_a[:800])


In [ ]:
print("─" * 60)
print("FULL — GrowthCo Daily Digest:")
print("─" * 60)
print(daily_digest_a)


In [ ]:
print("─" * 60)
print("FULL — RetailPro Daily Digest:")
print("─" * 60)
print(daily_digest_b)


---
## 14. Weekly Report Generation


In [ ]:
# ============================================================
# Section 14 — Weekly Report Generation
# ============================================================

FALLBACK_WEEKLY_A = """## 📊 Weekly Report — GrowthCo | Week of {date}

### 🎯 Executive Summary
This week's performance for GrowthCo reflects both the growth trajectory and the volatility characteristic of a startup in the paid acquisition phase. Revenue showed week-over-week changes driven primarily by paid channel performance. The evidence-based analysis below focuses on your top priority metrics.

### 📖 Weekly Narrative
The week began with stable performance across core acquisition metrics. Mid-week saw shifts in session volume correlated with changes in ad spend allocation. By week-end, revenue recovered partially from the mid-week dip. The conversion rate remained relatively stable throughout, suggesting that site-level factors were not the primary driver of revenue fluctuations.

### 🟢 Positive Trends
- **ROAS**: Week-over-week improvement indicates improving return on paid spend.
- **Repeat Purchase Rate**: Gradual upward trend over the 30-day window continues, which is healthy for a startup at this stage.
- **Email Revenue**: Consistent performance, contributing meaningfully to overall revenue mix.

### 🔴 Negative Trends
- **CAC**: Slight week-over-week increase. Based on the data, this correlates with higher CPM values observed this week.
- **Checkout Abandonment**: Still above 65% — the data does not reveal the specific cause, and further investigation is recommended.

### 🔀 Cross-Channel Analysis
- Paid (Meta + Google) remains the dominant session driver. Meta spend and session volume are correlated in the data.
- Email channel shows stable contribution. Open rates are within expected ranges.
- There is insufficient evidence in the current data to attribute the mid-week session dip to a specific channel.

### 💼 Business Implications
- The week's ROAS trajectory is encouraging for a growth-stage startup.
- CAC creep is worth monitoring — if it continues for another 2 weeks, it warrants a channel mix review.
- Checkout abandonment at current levels represents the most addressable conversion opportunity visible in the data.

### 🔧 Strategic Recommendations
1. **Audit checkout flow** — Abandonment at 65%+ is above industry benchmarks. The data does not reveal the specific friction point.
2. **Monitor CAC trend** — If week-over-week CAC increase persists next week, review bid strategies.
3. **Protect ROAS gains** — The improving ROAS trend should not be disrupted by rapid spend scaling without validation.

*Note: This report was generated without the Claude API. Set ANTHROPIC_API_KEY in .env to enable AI-written narratives.*
"""

FALLBACK_WEEKLY_B = """## 📊 Weekly Report — RetailPro | Week of {date}

### 🎯 Executive Summary
RetailPro's week was operationally significant. The data shows elevated pressure on inventory and fulfillment metrics, which are the top priorities for an established brand at this scale. Revenue performance was within expected ranges, but operational metrics require strategic attention.

### 📖 Weekly Narrative
The week's story centers on two operational themes: inventory depletion dynamics and the fulfillment SLA trend. Early in the week, inventory units continued their downward trajectory, which the data shows is correlated with sustained order volumes. By mid-week, fulfillment SLA days showed an upward movement that, if continued, will approach the breach threshold.

The repeat purchase rate remained strong, confirming the brand's retention advantage. However, the return rate moved above baseline — correlated in the data with the fulfillment SLA increase, though causation cannot be confirmed from available evidence alone.

### 🟢 Positive Trends
- **Repeat Purchase Rate**: Above 40% consistently — reflects RetailPro's retention strength.
- **Revenue**: Week-over-week performance is within expected range for this stage.
- **Email Open Rate**: Above-average open rates this week, indicating strong subscriber engagement.

### 🔴 Negative Trends
- **Inventory Units**: Declining trend continues. At current velocity, stock levels will reach critical threshold within an estimated 2-3 weeks based on the data trend.
- **Return Rate**: Above-baseline movement for the 4th consecutive day. Supporting data shows correlated increase in support tickets.
- **Fulfillment SLA Days**: Trending upward over 5 days. Still within threshold but approaching breach zone.

### 🔀 Cross-Channel Analysis
- Paid channels (Meta + Google) are performing within expected ranges and do not appear to be driving the operational anomalies.
- Email channel shows strong engagement but there is insufficient evidence to link it to the return rate increase.
- The inventory constraint, if it reaches critical levels, will impact fulfillment performance across all channels.

### 💼 Business Implications
- The convergence of declining inventory and rising SLA days represents an operational risk. Historical data shows this pattern has previously led to return rate spikes.
- Strong retention (repeat purchase rate) provides a buffer, but cannot offset the impact of fulfillment failures on brand trust.
- The current return rate trajectory, if sustained, will increase support costs and reverse the revenue trend.

### 🔧 Strategic Recommendations
1. **Prioritize inventory replenishment** — The data trend indicates critical stock levels within 2-3 weeks. Initiate procurement review immediately.
2. **Investigate return rate drivers** — Analyze support ticket categories for the past 7 days to identify the specific product or fulfillment issue.
3. **Monitor fulfillment SLA daily** — Set an alert threshold at 2.5 days to intervene before breach.
4. **Protect retention rate** — Any fulfillment failure that impacts the repeat purchase rate would be the highest-impact negative outcome.

*Note: This report was generated without the Claude API. Set ANTHROPIC_API_KEY in .env to enable AI-written narratives.*
"""


def call_claude_weekly(evidence: List[EvidencePacket], weekly_summary: Dict,
                        profile: CustomerProfile, report_date: str) -> str:
    """Call Claude API to generate a Weekly Report. Returns markdown string."""
    if not AI_ENABLED:
        print(f"   ⚠️  AI disabled — using fallback template for {profile.name}")
        fallback = FALLBACK_WEEKLY_A if profile.customer_id == 'A' else FALLBACK_WEEKLY_B
        return fallback.format(date=report_date)

    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    user_prompt = build_weekly_user_prompt(evidence, weekly_summary, profile, report_date)

    print(f"   🤖 Calling Claude ({CLAUDE_MODEL}) for {profile.name} weekly report...")
    response = client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=4000,
        temperature=0.3,
        system=WEEKLY_SYSTEM_PROMPT,
        messages=[
            {"role": "user", "content": user_prompt}
        ],
    )
    return response.content[0].text


print("🤖 Generating Weekly Reports...")
weekly_report_a = call_claude_weekly(evidence_a, weekly_summary_a, profile_a, REPORT_DATE_STR)
print(f"   ✅ GrowthCo weekly report: {len(weekly_report_a)} characters")

weekly_report_b = call_claude_weekly(evidence_b, weekly_summary_b, profile_b, REPORT_DATE_STR)
print(f"   ✅ RetailPro weekly report: {len(weekly_report_b)} characters")

print("\n" + "─" * 60)
print("PREVIEW — GrowthCo Weekly Report (first 600 chars):")
print("─" * 60)
print(weekly_report_a[:600])


In [ ]:
print("─" * 60)
print("FULL — GrowthCo Weekly Report:")
print("─" * 60)
print(weekly_report_a)


In [ ]:
print("─" * 60)
print("FULL — RetailPro Weekly Report:")
print("─" * 60)
print(weekly_report_b)


---
## 15. Export Reports


In [ ]:
# ============================================================
# Section 15 — Export Reports (Markdown + PDF)
# ============================================================

# ── Markdown Export ──────────────────────────────────────────

def save_markdown(content: str, path: Path) -> None:
    """Save report content as a Markdown file."""
    path.write_text(content, encoding='utf-8')
    print(f"   📄 Saved: {path}")


date_slug = REPORT_DATE_STR.replace('-', '')

# Daily digests
md_daily_a_path = DIRS['reports_daily'] / f"daily_digest_growthco_{date_slug}.md"
md_daily_b_path = DIRS['reports_daily'] / f"daily_digest_retailpro_{date_slug}.md"
save_markdown(daily_digest_a, md_daily_a_path)
save_markdown(daily_digest_b, md_daily_b_path)

# Weekly reports
md_weekly_a_path = DIRS['reports_weekly'] / f"weekly_report_growthco_{date_slug}.md"
md_weekly_b_path = DIRS['reports_weekly'] / f"weekly_report_retailpro_{date_slug}.md"
save_markdown(weekly_report_a, md_weekly_a_path)
save_markdown(weekly_report_b, md_weekly_b_path)

print("\n✅ Markdown exports complete.")


In [ ]:
# ── PDF Export ───────────────────────────────────────────────

class PDFReportGenerator:
    """
    Generates a professional PDF from a markdown-style report string.
    Uses ReportLab for layout control.
    """

    def __init__(self):
        self.styles = getSampleStyleSheet()
        self._add_custom_styles()

    def _add_custom_styles(self) -> None:
        """Define custom paragraph styles for the report."""
        self.styles.add(ParagraphStyle(
            name='ReportTitle',
            parent=self.styles['Title'],
            fontSize=22, textColor=colors.HexColor('#1E1B4B'),
            spaceAfter=12, fontName='Helvetica-Bold',
        ))
        self.styles.add(ParagraphStyle(
            name='ReportH1',
            parent=self.styles['Heading1'],
            fontSize=16, textColor=colors.HexColor('#3730A3'),
            spaceBefore=18, spaceAfter=8, fontName='Helvetica-Bold',
        ))
        self.styles.add(ParagraphStyle(
            name='ReportH2',
            parent=self.styles['Heading2'],
            fontSize=13, textColor=colors.HexColor('#4F46E5'),
            spaceBefore=14, spaceAfter=6, fontName='Helvetica-Bold',
        ))
        self.styles.add(ParagraphStyle(
            name='ReportBody',
            parent=self.styles['Normal'],
            fontSize=10, leading=16,
            textColor=colors.HexColor('#1E293B'),
            spaceAfter=6,
        ))
        self.styles.add(ParagraphStyle(
            name='ReportBullet',
            parent=self.styles['Normal'],
            fontSize=10, leading=16,
            leftIndent=20, firstLineIndent=-10,
            textColor=colors.HexColor('#1E293B'),
            spaceAfter=4,
        ))
        self.styles.add(ParagraphStyle(
            name='ReportMeta',
            parent=self.styles['Normal'],
            fontSize=9, textColor=colors.HexColor('#64748B'),
            spaceAfter=20, italic=True,
        ))

    def _markdown_to_flowables(self, text: str, chart_paths: Optional[List[Path]] = None) -> list:
        """Convert simple markdown to ReportLab flowables."""
        flowables = []

        for line in text.split('\n'):
            stripped = line.strip()
            if not stripped:
                flowables.append(Spacer(1, 4))
            elif stripped.startswith('## '):
                content = stripped[3:].replace('📋','').replace('📊','').strip()
                flowables.append(Paragraph(content, self.styles['ReportH1']))
                flowables.append(HRFlowable(width='100%', thickness=1.5,
                                             color=colors.HexColor('#4F46E5'), spaceAfter=8))
            elif stripped.startswith('### '):
                content = stripped[4:]
                for emoji in ['🎯','✅','⚠️','📈','📊','🔧','📖','🟢','🔴','🔀','💼']:
                    content = content.replace(emoji, '').strip()
                flowables.append(Paragraph(content, self.styles['ReportH2']))
            elif stripped.startswith('- **'):
                # Bold bullet
                content = stripped[2:]
                content = content.replace('**', '<b>', 1).replace('**', '</b>', 1)
                flowables.append(Paragraph(f"• {content}", self.styles['ReportBullet']))
            elif stripped.startswith('- '):
                content = stripped[2:]
                flowables.append(Paragraph(f"• {content}", self.styles['ReportBullet']))
            elif stripped.startswith(('1.','2.','3.','4.','5.')):
                flowables.append(Paragraph(stripped, self.styles['ReportBullet']))
            elif stripped.startswith('*Note:'):
                flowables.append(Paragraph(stripped, self.styles['ReportMeta']))
            elif stripped.startswith('---'):
                flowables.append(HRFlowable(width='100%', thickness=0.5,
                                             color=colors.HexColor('#CBD5E1'), spaceAfter=8))
            elif stripped:
                flowables.append(Paragraph(stripped, self.styles['ReportBody']))

        # Embed charts if provided
        if chart_paths:
            flowables.append(PageBreak())
            flowables.append(Paragraph("Appendix: Supporting Charts", self.styles['ReportH1']))
            for cp in chart_paths:
                if cp.exists():
                    try:
                        img = RLImage(str(cp), width=6.5*inch, height=2.7*inch)
                        flowables.append(img)
                        flowables.append(Spacer(1, 12))
                    except Exception as e:
                        flowables.append(Paragraph(f"[Chart: {cp.name}]", self.styles['ReportMeta']))

        return flowables

    def generate(self, content: str, output_path: Path,
                  title: str, customer_name: str,
                  chart_paths: Optional[List[Path]] = None) -> None:
        """Generate a PDF report."""
        doc = SimpleDocTemplate(
            str(output_path),
            pagesize=A4,
            rightMargin=2*cm, leftMargin=2*cm,
            topMargin=2.5*cm, bottomMargin=2.5*cm,
            title=title, author="AI Report Generation System",
        )

        flowables = []

        # Cover header
        flowables.append(Paragraph(title, self.styles['ReportTitle']))
        flowables.append(Paragraph(
            f"Customer: {customer_name} | Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}",
            self.styles['ReportMeta']
        ))
        flowables.append(HRFlowable(width='100%', thickness=2,
                                     color=colors.HexColor('#4F46E5'), spaceAfter=20))

        flowables.extend(self._markdown_to_flowables(content, chart_paths))
        doc.build(flowables)
        print(f"   📑 PDF saved: {output_path}")


pdf_gen = PDFReportGenerator()

print("📑 Generating PDF reports...")

# Daily PDFs
pdf_gen.generate(
    daily_digest_a,
    DIRS['reports_daily'] / f"daily_digest_growthco_{date_slug}.pdf",
    title=f"Daily Digest — GrowthCo | {REPORT_DATE_STR}",
    customer_name="GrowthCo",
    chart_paths=[CHART_PATHS.get('revenue_trend_a'), CHART_PATHS.get('roas_trend'),
                 CHART_PATHS.get('importance_a')]
)

pdf_gen.generate(
    daily_digest_b,
    DIRS['reports_daily'] / f"daily_digest_retailpro_{date_slug}.pdf",
    title=f"Daily Digest — RetailPro | {REPORT_DATE_STR}",
    customer_name="RetailPro",
    chart_paths=[CHART_PATHS.get('inventory_fulfillment_b'), CHART_PATHS.get('importance_b')]
)

# Weekly PDFs
pdf_gen.generate(
    weekly_report_a,
    DIRS['reports_weekly'] / f"weekly_report_growthco_{date_slug}.pdf",
    title=f"Weekly Report — GrowthCo | {REPORT_DATE_STR}",
    customer_name="GrowthCo",
    chart_paths=[CHART_PATHS.get('revenue_trend_a'), CHART_PATHS.get('revenue_comparison'),
                 CHART_PATHS.get('roas_trend'), CHART_PATHS.get('cac_trend'),
                 CHART_PATHS.get('weekly_revenue_a'), CHART_PATHS.get('zscore_revenue_a')]
)

pdf_gen.generate(
    weekly_report_b,
    DIRS['reports_weekly'] / f"weekly_report_retailpro_{date_slug}.pdf",
    title=f"Weekly Report — RetailPro | {REPORT_DATE_STR}",
    customer_name="RetailPro",
    chart_paths=[CHART_PATHS.get('inventory_fulfillment_b'), CHART_PATHS.get('corr_heatmap'),
                 CHART_PATHS.get('importance_b')]
)

print("\n✅ All reports exported.")

# Summary
all_reports = list(DIRS['reports_daily'].glob('*')) + list(DIRS['reports_weekly'].glob('*'))
print(f"\n📁 Total report files: {len(all_reports)}")
for r in sorted(all_reports):
    size = r.stat().st_size
    print(f"   {r.name:<60} {size/1024:>6.1f} KB")


---
## 16. Product Design Documentation


In [ ]:
# ============================================================
# Section 16 — Product Design Documentation
# ============================================================

PRODUCT_DESIGN_DOC = f"""# Product Design Document
# AI-Powered Ecommerce Report Generation System

> Version 1.0 | {datetime.now().strftime('%Y-%m-%d')} | Generated by notebook.ipynb

---

## 1. Users

### Primary Users
| User Type | Description | Key Need |
|-----------|-------------|----------|
| **Founder / CEO** | D2C brand owner, 1-10 person team | "What happened today? Do I need to act?" |
| **Head of Growth** | Performance marketer, ROAS-obsessed | "Is my paid spend working?" |
| **Head of Operations** | Inventory, fulfillment, CS | "Are we operationally healthy?" |
| **Investor / Board** | Wants weekly business narrative | "What's the trend?" |

### Customer Archetypes Prototyped
| | GrowthCo (A) | RetailPro (B) |
|---|---|---|
| Stage | Startup | Established |
| Size | Small | Medium |
| Focus | Acquisition | Retention & Operations |
| Top metric | Revenue, ROAS | Inventory, Returns |
| Personalization | Warm (45 events) | Mature (120 events) |

---

## 2. Jobs-to-be-Done (JTBD)

### Daily Digest
- **JTBD**: "When I wake up, I want to know if anything significant changed in my business overnight, so I can prioritize my day."
- **Success criterion**: User can read in <5 minutes and knows exactly what to do (or that everything is fine).
- **Anti-JTBD**: Not a dashboard replacement. The user should NOT need to click through charts.

### Weekly Report
- **JTBD**: "On Monday morning, I want a narrative of last week's business performance to share with my team and investors."
- **Success criterion**: User can share the report with a non-technical stakeholder with confidence.
- **Anti-JTBD**: Not a data dump. Every metric must contribute to a coherent story.

---

## 3. Daily Digest vs. Weekly Report

| Dimension | Daily Digest | Weekly Report |
|-----------|-------------|---------------|
| Time horizon | Yesterday vs. today | Last 7 days vs. prior 7 days |
| Reading time | ~5 minutes | ~15 minutes |
| Primary question | "What changed?" | "What's the story?" |
| Tone | Alert-style, actionable | Narrative, analytical |
| Audience | Operator | Operator + Stakeholder |
| Top metric focus | Today's anomalies | 7-day trends |
| Chart depth | 2-3 key charts | 5-6 trend charts |

---

## 4. Importance Ranking Strategy

### Design Philosophy
The ranking engine is **deterministic** (no ML) to ensure:
- Full auditability: every score has a human-readable breakdown
- No training data required
- Instant deployment for new customers
- Customer trust ("why is this #1?" is always answerable)

### Scoring Formula
```
ImportanceScore(0-100) =
    35% × BusinessImportance    (customer priority weights)
  + 25% × StatisticalSurprise  (Z-score normalized, sigmoid)
  + 20% × Magnitude            (absolute % change, capped at 50%)
  + 10% × TrendPersistence     (consecutive days same direction)
  +  5% × Novelty              (not seen recently = higher)
  +  5% × BusinessRules        (safety nets for always-important metrics)
```

### Safety Nets (Business Rules)
| Metric | Rule | Why |
|--------|------|-----|
| Revenue | Always in top 5 | Revenue is never trivial |
| ROAS | Boost when < 2.0 | Unit economics survival |
| Inventory | Boost when < 10% of baseline | Stockout risk |
| Fulfillment SLA | Boost when > threshold | Customer experience SLA breach |

---

## 5. Personalization Strategy

### Three-Tier System

```
Cold Start (0-4 weeks)
  → Industry defaults + company stage multipliers
  → No personalization adjustment

Warm (4-12 weeks)
  → Moderate adjustment from click/ignore patterns
  → Clicked metrics: +15% score boost per engagement
  → Ignored metrics (2:1 ratio): -15% score decay

Mature (3+ months)
  → Fully learned per-metric click rates
  → Score multiplier = 0.7 + 0.6 × click_rate
  → Priority metrics always receive +25% boost
```

### Cold Start Solution
For new customers, the system collects during onboarding:
1. Industry (determines base weights)
2. Company stage (startup / growth / established / enterprise)
3. Self-reported top 3 metrics
These three inputs fully initialize the system without any historical data.

---

## 6. System Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│                        DATA LAYER                               │
│  Ecommerce Platform APIs (Shopify, WooCommerce, BigCommerce)    │
│  → Raw event + transaction data → Daily CSV/Parquet snapshots   │
└───────────────────────────────┬─────────────────────────────────┘
                                │
┌───────────────────────────────▼─────────────────────────────────┐
│                     ANALYTICS ENGINE                            │
│  Feature Engineering → Movement Detection → Importance Ranking  │
│  (Deterministic, fully auditable, reproducible)                 │
└───────────────────────────────┬─────────────────────────────────┘
                                │
┌───────────────────────────────▼─────────────────────────────────┐
│                  PERSONALIZATION LAYER                          │
│  Customer Profile → Tier Selection → Weight Adjustment          │
│  (Cold Start / Warm / Mature)                                   │
└───────────────────────────────┬─────────────────────────────────┘
                                │
┌───────────────────────────────▼─────────────────────────────────┐
│                   EVIDENCE BUILDER                              │
│  Converts ranked movements → Structured JSON packets            │
│  (LLM never sees raw data)                                      │
└───────────────────────────────┬─────────────────────────────────┘
                                │
┌───────────────────────────────▼─────────────────────────────────┐
│                      LLM LAYER (Claude)                         │
│  System Prompt + Evidence JSON → AI-written narrative           │
│  (Strictly grounded, temperature=0.3)                           │
└───────────────────────────────┬─────────────────────────────────┘
                                │
┌───────────────────────────────▼─────────────────────────────────┐
│                    DELIVERY LAYER                               │
│  Markdown + PDF generation → Email / Slack / API delivery       │
└─────────────────────────────────────────────────────────────────┘
```

### Batch vs. Online Processing
| Concern | Decision | Rationale |
|---------|----------|-----------|
| Data ingestion | **Batch** (nightly) | Ecommerce data lags 2-4h; real-time adds cost without value |
| Feature engineering | **Batch** | Rolling stats need historical context; compute offline |
| Ranking engine | **Batch** | Fully deterministic; precompute at ingestion time |
| LLM generation | **Online** (at send time) | Allows late-breaking context injection; lower stale risk |
| Report delivery | **Scheduled** (6am customer TZ) | Digest reads best at day start |

### LLM Placement
The LLM is placed **after** the full analytics pipeline — not at the beginning.
This is a critical architectural decision:
- The LLM **cannot change** which metrics are selected (deterministic engine owns that)
- The LLM **cannot hallucinate** correlations (evidence builder constrains the context)
- The LLM **only writes prose** — it is a summarization layer, not an analysis layer

### Deterministic vs. Generative Separation
```
DETERMINISTIC (engineered):          GENERATIVE (LLM):
- Metric selection                   - Narrative prose
- Importance ranking                 - Tone adaptation
- Anomaly detection                  - Report structure
- Evidence construction              - Sentence variety
- Statistical confidence             - Contextual phrasing
```

---

## 7. Failure Modes & Mitigations

*(See Section 17 for detailed analysis)*

| Failure Mode | Severity | Mitigation |
|---|---|---|
| LLM Hallucination | HIGH | Evidence-only prompting, strict system prompt, temperature=0.3 |
| False Importance | HIGH | Conservative Z-score thresholds, business rule safety nets |
| Cold Start Quality | MEDIUM | Industry defaults + onboarding questionnaire |
| Metric Overload | MEDIUM | Hard cap: top 8 metrics per report |
| Noise as Signal | MEDIUM | 30-day rolling baseline, dual Z-score (7d + 30d) |
| Incorrect Attribution | HIGH | Evidence builder only includes directly observed correlations |

---

## 8. Product Success Metrics

These are **product metrics** — not LLM quality metrics.

| Metric | Definition | Target |
|--------|------------|--------|
| **Digest Open Rate** | % of delivered digests opened within 2h | >70% |
| **Completion Rate** | % of opened reports read to the end | >60% |
| **Action Rate** | % of reports where user takes a recommended action | >30% |
| **Report Usefulness Score** | User 1-5 rating ("Was this useful?") | >4.0 avg |
| **Ignored Insights %** | % of top-ranked insights the user consistently ignores | <20% |
| **Follow-up Question Rate** | % of reports that generate a user question | >15% |
| **Personalization Lift** | Score improvement after 4 weeks vs. cold start | +15% action rate |
| **Churn Correlation** | Reports opened in last 7 days as a churn signal | Lagging indicator |

### North Star Metric
> **Action Rate** — A report is only valuable if it changes what the operator does that day.

---

*End of Product Design Document*
"""

# Save as markdown
pdd_path = BASE_DIR / 'PRODUCT_DESIGN.md'
pdd_path.write_text(PRODUCT_DESIGN_DOC, encoding='utf-8')
print(f"✅ Product Design Document saved: {pdd_path}")
print(f"   {len(PRODUCT_DESIGN_DOC)} characters, {len(PRODUCT_DESIGN_DOC.splitlines())} lines")


---
## 17. Failure Modes Analysis


In [ ]:
# ============================================================
# Section 17 — Failure Modes Analysis
# ============================================================

FAILURE_MODES_DOC = """
# Failure Modes Analysis
## AI-Powered Ecommerce Report Generation System

---

## Failure Mode 1: LLM Hallucination

### Description
The LLM invents causal explanations, references external events (competitor launches,
algorithm changes), or fabricates supporting data that wasn't provided.

### Example
Revenue is down 12%. LLM writes: "This is likely due to Meta's algorithm change announced
last week." — but we never provided this information.

### Mitigation Strategies
1. **Evidence-only prompting**: LLM receives structured JSON, never free-text or raw tables
2. **Explicit grounding instruction**: System prompt states "Only reference data in evidence JSON"
3. **Low temperature (0.3)**: Reduces creative generation
4. **Insufficient evidence clause**: Prompt instructs LLM to say "insufficient evidence" when uncertain
5. **Post-generation validation** (production): Parse output, verify all numbers match evidence
6. **Human review gate**: For CRITICAL severity reports, flag for human review before delivery

### Residual Risk
Low-to-medium. Well-prompted Claude 3+ models hallucinate rarely on structured data.
Risk is higher for abstract causal claims than for number accuracy.

---

## Failure Mode 2: False Importance (Wrong Metrics Surface)

### Description
A metric with a high Z-score receives a high importance score, but the underlying
cause is a data pipeline error, not a real business event.

### Example
A data lag causes revenue to show as $0 for 2 hours. The Z-score is 5.0 (CRITICAL).
The report leads with a false alarm.

### Mitigation Strategies
1. **Data freshness checks**: Flag and quarantine metrics with staleness > 2h before ranking
2. **Plausibility bounds**: Hard reject values outside physical bounds (revenue < 0, CVR > 100%)
3. **Multi-signal confirmation**: CRITICAL severity requires Z-score AND daily % change > 20%
4. **Historical anomaly baseline**: Use 30-day rolling Z-score, not 7-day (more stable)
5. **Confidence score display**: Show confidence level to user alongside finding

### Residual Risk
Medium. Data pipeline quality is an upstream dependency. In production, this requires
a separate data quality monitoring system.

---

## Failure Mode 3: Cold Start Quality Degradation

### Description
A brand-new customer receives a report that is poorly personalized because the system
has no interaction history and the industry defaults don't match their actual business.

### Example
A DTC supplements brand is mapped to the "fashion" defaults, resulting in ROAS and
CTR being over-weighted when they actually care more about subscription retention.

### Mitigation Strategies
1. **Onboarding questionnaire**: Collect industry, stage, and top 3 self-reported metrics
2. **Broader industry taxonomy**: Expand from 2 to 20+ industry defaults
3. **Explicit weight override**: Let customers adjust weights in the first week via UI
4. **Rapid learning loop**: Accelerate weight adjustment for the first 10 interactions
5. **Fallback messaging**: In cold-start reports, add: "Personalization improves with use"

### Residual Risk
Low. Cold start quality is bounded — it's never worse than a reasonable industry default.

---

## Failure Mode 4: Metric Overload

### Description
Too many metrics surface in the report, overwhelming the reader. Instead of answering
"What changed?", the report becomes a data dump.

### Example
All 23 metrics show movement on a volatile day. The report lists all 23 with equal prominence.

### Mitigation Strategies
1. **Hard cap**: Never include more than 8 metrics in a Daily Digest
2. **Section limits**: Max 3 metrics in "Attention Required", 2 in "Top Wins"
3. **Importance threshold**: Only include metrics with ImportanceScore > 40
4. **Severity filter**: Surface CRITICAL and HIGH first; MEDIUM and LOW only if slots remain
5. **Supporting Metrics section**: Group low-importance metrics in a compact table

### Residual Risk
Low. Hard caps are enforced by the Evidence Builder (top_n=8).

---

## Failure Mode 5: Noise Mistaken as Signal

### Description
Daily random variation in a noisy metric triggers a HIGH or CRITICAL alert,
causing alert fatigue and eroding trust in the system.

### Example
Email open rate fluctuates ±5% daily. On a high-noise day, it spikes 8% and receives
a HIGH severity tag, appearing as a "Top Win" when it's just normal variation.

### Mitigation Strategies
1. **Volatility-adjusted Z-scores**: High-volatility metrics require larger moves to trigger
2. **Confidence score**: Low-confidence signals (high volatility) are labeled as such
3. **Trend persistence requirement**: Emerging Signals section requires 3+ consecutive days
4. **Metric-specific thresholds**: Email open rate needs Z > 2.5 vs. revenue's Z > 2.0
5. **User feedback loop**: "Ignore this metric" button reduces future weight permanently

### Residual Risk
Medium. Inherent in any anomaly detection system. Volatility adjustment reduces false
positives significantly but cannot eliminate them entirely.

---

## Failure Mode 6: Incorrect Causal Attribution

### Description
The system correctly detects that Revenue is down, and correctly identifies that
Sessions are also down — but implies causation when it can only show correlation.

### Example
"Revenue declined 12% because sessions dropped 15%." — Sessions declined for the
same reason (Meta outage), not because of a chain of causation.

### Mitigation Strategies
1. **Correlation language only**: Evidence packets use "is correlated with" not "caused by"
2. **Supporting facts framing**: "Sessions also declined" — never "sessions caused the decline"
3. **Explicit uncertainty**: System prompt instructs Claude to say "The cause is unclear"
4. **Causal chain limits**: Evidence builder caps at 4 supporting facts to prevent false narratives
5. **User education**: Onboarding explains the distinction between correlation and causation

### Residual Risk
Medium-low. The LLM may still imply causation in natural language despite instructions.
Post-generation NLP validation can catch explicit causal language as a safety check.

---

*Failure Mode Analysis — End*
"""

print(FAILURE_MODES_DOC)


---
## 18. Evaluation Framework


In [ ]:
# ============================================================
# Section 18 — Evaluation Framework
# ============================================================
# NOTE: These are PRODUCT metrics — not LLM quality metrics.
# They measure business value delivered, not text quality.
# ============================================================

# Simulate historical product engagement data for evaluation
rng_eval = np.random.default_rng(42)
N_EVAL_DAYS = 30

# Simulate for both customers
eval_records = []
for cid, name, tier in [('A', 'GrowthCo', 'warm'), ('B', 'RetailPro', 'mature')]:
    for day in range(N_EVAL_DAYS):
        # Mature customers engage more
        base_open = 0.75 if tier == 'mature' else 0.60
        opened = rng_eval.random() < base_open
        if opened:
            completion = rng_eval.random() < (0.70 if tier == 'mature' else 0.55)
            action     = rng_eval.random() < (0.40 if tier == 'mature' else 0.25)
            usefulness = rng_eval.choice([3, 4, 4, 5, 5]) if tier == 'mature' else rng_eval.choice([2, 3, 4, 4])
            follow_up  = rng_eval.random() < 0.20
        else:
            completion = False
            action     = False
            usefulness = None
            follow_up  = False

        eval_records.append({
            'customer_id': cid, 'customer_name': name, 'tier': tier,
            'day': day, 'report_type': 'daily',
            'opened': opened, 'completed': completion,
            'action_taken': action, 'follow_up': follow_up,
            'usefulness_score': usefulness,
        })

eval_df = pd.DataFrame(eval_records)

# Compute product metrics
def compute_product_metrics(df: pd.DataFrame, customer_id: str) -> Dict:
    """Compute product engagement metrics for a customer."""
    sub = df[df['customer_id'] == customer_id]
    opened    = sub[sub['opened']]
    completed = sub[sub['completed']]
    acted     = sub[sub['action_taken']]
    rated     = sub[sub['usefulness_score'].notna()]

    return {
        'open_rate':          len(opened) / len(sub) if len(sub) else 0,
        'completion_rate':    len(completed) / len(opened) if len(opened) else 0,
        'action_rate':        len(acted) / len(opened) if len(opened) else 0,
        'avg_usefulness':     rated['usefulness_score'].mean() if len(rated) else 0,
        'follow_up_rate':     sub['follow_up'].mean(),
        'total_reports':      len(sub),
        'reports_opened':     len(opened),
    }

metrics_a_eval = compute_product_metrics(eval_df, 'A')
metrics_b_eval = compute_product_metrics(eval_df, 'B')

print("📊 Product Engagement Metrics (Simulated 30-Day Period)")
print("\n" + "─" * 65)
print(f"{'Metric':<30} {'GrowthCo (Warm)':>16} {'RetailPro (Mature)':>18}")
print("─" * 65)

display_metrics = [
    ('Open Rate',          'open_rate',        '{:.1%}'),
    ('Completion Rate',    'completion_rate',   '{:.1%}'),
    ('Action Rate',        'action_rate',       '{:.1%}'),
    ('Avg Usefulness (1-5)','avg_usefulness',   '{:.2f}'),
    ('Follow-up Rate',     'follow_up_rate',    '{:.1%}'),
    ('Total Reports',      'total_reports',     '{:.0f}'),
]

for label, key, fmt in display_metrics:
    va = fmt.format(metrics_a_eval[key])
    vb = fmt.format(metrics_b_eval[key])
    print(f"{label:<30} {va:>16} {vb:>18}")

print("─" * 65)
print("\n💡 Key observations:")
print(f"   • Mature tier (RetailPro) shows +{(metrics_b_eval['open_rate'] - metrics_a_eval['open_rate'])*100:.1f}pp open rate lift")
print(f"   • Action rate improves with tier maturity — personalization is working")
print(f"   • Follow-up questions are a positive signal (customer engagement)")


In [ ]:
# ── Chart 10: Product Evaluation Metrics ────────────────────
categories = ['Open Rate', 'Completion Rate', 'Action Rate', 'Follow-up Rate']
vals_a = [metrics_a_eval['open_rate'], metrics_a_eval['completion_rate'],
           metrics_a_eval['action_rate'], metrics_a_eval['follow_up_rate']]
vals_b = [metrics_b_eval['open_rate'], metrics_b_eval['completion_rate'],
           metrics_b_eval['action_rate'], metrics_b_eval['follow_up_rate']]

fig = go.Figure(data=[
    go.Bar(name='GrowthCo (Warm)',    x=categories, y=[v*100 for v in vals_a],
            marker_color=COLORS['customerA'], text=[f"{v:.0%}" for v in vals_a], textposition='outside'),
    go.Bar(name='RetailPro (Mature)', x=categories, y=[v*100 for v in vals_b],
            marker_color=COLORS['customerB'], text=[f"{v:.0%}" for v in vals_b], textposition='outside'),
])

fig.update_layout(
    barmode='group',
    title='Product Engagement Metrics — Warm vs. Mature Customer (Simulated 30 Days)',
    yaxis_title='Rate (%)', yaxis_range=[0, 100],
    height=CHART_HEIGHT, width=CHART_WIDTH,
)
CHART_PATHS['eval_metrics'] = save_plotly_chart(fig, 'eval_metrics')
fig.show()


In [ ]:
# ── Final Summary ────────────────────────────────────────────
print("=" * 70)
print("  AI-POWERED ECOMMERCE REPORT GENERATION SYSTEM")
print("  End-to-End Run Complete")
print("=" * 70)

print("\n📊 Data Generated:")
print(f"   ecommerce_data.csv : {len(df_all)} rows | {len(df_all.columns)} columns")
print(f"   customers.csv      : {len(customers_data)} customer profiles")
print(f"   Feature columns    : {df_a_feat.shape[1]} (incl. engineered features)")

print("\n🔍 Analysis:")
print(f"   Movements detected : {len(movements_a)} (GrowthCo) | {len(movements_b)} (RetailPro)")
print(f"   Ranked scores      : {len(ranked_a_pers)} (GrowthCo) | {len(ranked_b_pers)} (RetailPro)")
print(f"   Evidence packets   : {len(evidence_a)} (GrowthCo) | {len(evidence_b)} (RetailPro)")

print("\n🤖 Reports Generated:")
print(f"   Daily Digest A     : {len(daily_digest_a)} chars ({len(daily_digest_a)//200} min read est.)")
print(f"   Daily Digest B     : {len(daily_digest_b)} chars ({len(daily_digest_b)//200} min read est.)")
print(f"   Weekly Report A    : {len(weekly_report_a)} chars ({len(weekly_report_a)//200} min read est.)")
print(f"   Weekly Report B    : {len(weekly_report_b)} chars ({len(weekly_report_b)//200} min read est.)")

print("\n📄 Exported Files:")
all_files = (
    list(DIRS['reports_daily'].glob('*')) +
    list(DIRS['reports_weekly'].glob('*')) +
    list(DIRS['charts'].glob('*.png')) +
    [BASE_DIR / 'PRODUCT_DESIGN.md']
)
for f in sorted(all_files):
    if f.exists():
        size = f.stat().st_size
        print(f"   ✓ {str(f.relative_to(BASE_DIR)):<55} {size/1024:>6.1f} KB")

print("\n✅ All done. Notebook run complete.")
print("=" * 70)
